<!-- COMMONS LAUNCHER v4 · generated by tools/notebooks.py · do not edit by hand -->
<a href="https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials"><img src="https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/brand/synapsa-commons-badge.png" alt="Synapsa Commons" height="36"></a>

Free, hands-on AI courses that run anywhere, from the team building [Synapsa](https://synapsa.realai.eu), an AI-native
learning platform.

© 2026 RealAI · free to learn from, share and adapt, not to sell ([CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)).
The notice at the end of this notebook says what you may and may not do.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/lessons/T04-L06-dpo-from-scratch/lesson.ipynb)
[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/lessons/T04-L06-dpo-from-scratch/lesson.ipynb)
[![Open in Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master?labpath=lessons/T04-L06-dpo-from-scratch/lesson.ipynb)
[![Open in Codespaces](https://github.com/codespaces/badge.svg)](https://codespaces.new/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials)

This lesson needs Python 3.11 or newer with numpy, which Colab, Kaggle, Binder and
Codespaces already have.

In [ ]:
# --- COMMONS LAUNCHER v4 · generated by tools/notebooks.py · do not edit by hand ---
# Makes this notebook run anywhere. Every line is a no-op when the thing is already present,
# so a local clone pays nothing and an online notebook repairs itself.
import importlib.util, os, subprocess, sys, urllib.request
from pathlib import Path

COMMONS_PIP = []            # (import name, pinned pip spec) for what this lesson imports
COMMONS_SIBLINGS = []    # files that must sit beside the notebook
# A fork, a classroom mirror or an offline copy can serve the files from elsewhere by setting
# COMMONS_RAW_OVERRIDE before running this cell.
COMMONS_RAW = os.environ.get("COMMONS_RAW_OVERRIDE") or "https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/lessons/T04-L06-dpo-from-scratch/"

# Resolve siblings against the LESSON's own directory, not the working directory. A notebook
# has no __file__ and runs with cwd alongside itself; a grader imports this file from the repo
# root. Checking cwd blindly makes the grader think every sibling is missing and reach for the
# network -- which would put a download on a graded path.
try:
    COMMONS_DIR = Path(__file__).resolve().parent
except NameError:
    COMMONS_DIR = Path.cwd()


def commons_host() -> str:
    """Name the notebook service we are on. Used for the message, and for honest errors."""
    try:
        if importlib.util.find_spec("google.colab") is not None:
            return "Google Colab"
    except (ImportError, ValueError):
        pass
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
        return "Kaggle"
    if os.environ.get("BINDER_SERVICE_HOST"):
        return "Binder"
    if os.environ.get("CODESPACES"):
        return "GitHub Codespaces"
    return "a local Python environment"


_missing = [pip for imp, pip in COMMONS_PIP if importlib.util.find_spec(imp) is None]
if _missing:
    print("installing " + ", ".join(_missing) + " ...")
    # pip everywhere a student is likely to be; uv-managed local venvs ship without pip.
    if importlib.util.find_spec("pip") is not None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_missing], check=True)
    else:
        subprocess.run(["uv", "pip", "install", "-q", "--python", sys.executable, *_missing],
                       check=True)
    importlib.invalidate_caches()

_fetched = []
for _name in COMMONS_SIBLINGS:
    if not (COMMONS_DIR / _name).exists():
        (COMMONS_DIR / _name).parent.mkdir(parents=True, exist_ok=True)
        try:
            urllib.request.urlretrieve(COMMONS_RAW + _name, COMMONS_DIR / _name)
            _fetched.append(_name)
        except Exception as _e:  # Kaggle disables the internet by default; say so plainly
            raise RuntimeError(
                f"this lesson needs {_name} beside the notebook and could not fetch it "
                f"({_e}). On Kaggle, switch Internet on in the notebook settings panel "
                f"(Kaggle allows that only for phone-verified accounts); otherwise download it "
                f"from {COMMONS_RAW + _name} and upload it beside the notebook."
            ) from None

print("ready on " + commons_host() + ("; fetched " + ", ".join(_fetched) if _fetched else ""))
# --- END COMMONS LAUNCHER ---

# T04-L06 · DPO from the RLHF objective: derive the loss, implement it on the nanolm, and measure what it does

**You will build:** Direct Preference Optimisation (DPO) on a small language model, from the
objective up. You derive the loss from the KL-regularised reward objective, implement it in numpy on a
compact copy of the nanolm of T03-L03, check its gradient against finite differences, and then
measure what it does: to the margin it was built to raise, to the likelihood of the answer you
told it to prefer, to the length of what the model writes, and to its distance from the model
it started as.

**Time:** ~2.5 hours · **Runs on:** a laptop CPU, no GPU, no download, no network ·
**Prerequisites:** T03-L03-nanolm-cpu-slice (the model and its hand-derived backward pass, which
this notebook carries as a compact copy and does not load), T04-L02-chat-templates-and-masking
(why only the response tokens count in the loss).

By the end you will be able to:

1. Follow the derivation of the DPO loss from the KL-regularised reward objective, and check numerically that the
   closed-form optimum maximises the objective and that the partition function cancels in a pair.
2. Implement the stable log-likelihood of a response, the DPO loss with its implicit reward
   margins, its gradient coefficients, and the backward pass that re-weights per-sequence
   log-likelihood gradients; verify the whole gradient against finite differences.
3. Measure, on held-out prompts, the implicit reward margin and pair accuracy of a DPO-trained
   model, and count the pairs where the preferred answer became less likely while the margin rose.
4. Audit a preference set for a length bias with a length-matched comparison and a length-blind control, and measure how much
   longer, and how much more correct, a DPO-trained model's answers are than the reference's and
   than those of SFT on the preferred answers alone.
5. Implement a KL estimate with its standard error and the IPO loss, and measure how beta and the
   choice of loss change the distance the policy travels.

**This is not a real LLM.** The model has a few thousand parameters and a ten-symbol vocabulary,
and the "assistant" does one thing. The lesson says where that changes what you can conclude.

<!-- COMMONS BEFORE v1 · generated from meta.yaml by tools/notebooks.py · do not edit by hand -->
## Before you start

**Lessons to do first:** `T03-L03`, `T04-L02`

**This lesson assumes you can already:**

- write numpy code on arrays of shape (batch, time, vocabulary) and take a softmax or a sum along the last axis
- say what the log-likelihood of a sequence is, and why the probabilities of its tokens multiply
- read a KL divergence between two distributions and say what it measures
- apply the chain rule to a composed function (T03-L03 derives the model's backward pass this lesson reuses)

**New to some of that? These are good places to learn it** (free to read online first, then paid books). You do not need any of them if the list above already reads as familiar.

- [Direct Preference Optimization: Your Language Model is Secretly a Reward Model](https://arxiv.org/abs/2305.18290) · Rafailov et al. (arXiv 2305.18290) · *free* — section 4 is the derivation of sections 6 to 9 of this lesson (the closed-form optimum, the cancelled partition function, the loss and its gradient), and appendix A has the long form
- [A General Theoretical Paradigm to Understand Learning from Human Preferences](https://arxiv.org/abs/2310.12036) · Azar et al. (arXiv 2310.12036) · *free* — section 5.2 derives the IPO loss of exercise 8, and section 4.2 gives the argument for why DPO's logistic loss has no finish line
- [Illustrating Reinforcement Learning from Human Feedback (RLHF)](https://huggingface.co/blog/rlhf) · Hugging Face blog · *free* — the pipeline of reward model, KL penalty and policy that section 6 starts from, in words and pictures before the algebra
- [DPO Trainer](https://huggingface.co/docs/trl/dpo_trainer) · Hugging Face TRL documentation · *free* — how a production library exposes the same loss and its beta, to compare with the 20 lines you write here
- [NumPy, the absolute basics for beginners](https://numpy.org/doc/stable/user/absolute_beginners.html) · The NumPy project · *free* — arrays, axes and the few operations the eight exercises lean on

In [ ]:
# Setup: everything the lesson needs, in one cell, with versions printed.
import contextlib
import io
import sys
import traceback
from typing import Callable

import numpy as np

print("python", sys.version.split()[0], "· numpy", np.__version__)

SEED = 20261102             # this lesson's one fixed seed: every random stream derives from it
N_LETTERS = 6               # letters a..f
PROMPT_LEN = 4              # a prompt is four letters
MAX_RESP = 14               # a response is cut off after this many tokens
WIDTH = PROMPT_LEN + 1 + MAX_RESP   # tokens in one padded sequence: prompt, separator, response
CTX = WIDTH - 1             # positions the model sees (it predicts the next token at each)
VOCAB = 10                  # six letters, separator, end, filler, padding
SEP, EOS, FILL, PAD = 6, 7, 8, 9
CHARS = "abcdef|.x_"        # how a token id prints
D_MODEL = 32
P_WRONG = 0.3               # chance that the reference's training answers get one letter wrong
MAX_FILL = 5                # the reference's training answers carry at most this many filler tokens
LENGTH_BONUS = 0.4          # the PLANTED bias: the labeller adds this per response token
N_TRAIN_PROMPTS = 1000
PAIRS_PER_PROMPT = 3
REF_STEPS = 1500
REF_LR = 3e-3
PAIR_BATCH = 64
POLICY_STEPS = 300
POLICY_LR = 3e-4
BETA = 0.1
SEEDS = (0, 1, 2)           # training seeds for the main comparison

_FAILED_CHECKS: list[str] = []
_STATUS: dict[str, str] = {}   # label -> "passed" | "failed" | "not started", latest run

# The exercises, in the order you meet them, and the functions each one asks you to write.
# The progress board at the foot of the notebook is built from this, and a cell that is
# waiting on an unfinished exercise names it from here.
_EXERCISES: dict[str, tuple[str, ...]] = {
    "exercise 1": ("win_counts",),
    "exercise 2": ("sequence_logprob",),
    "exercise 3": ("dpo_loss",),
    "exercise 4": ("dpo_coefficients",),
    "exercise 5": ("sequence_logprob_backward",),
    "exercise 6": ("displacement_counts",),
    "exercise 7": ("kl_estimate",),
    "exercise 8": ("ipo_loss",),
}


def _named(labels: list[str]) -> str:
    """["exercise 3"] -> "exercise 3 (dpo_loss)"; several -> "exercises 2, 3 and 4"."""
    if len(labels) == 1:
        return f"{labels[0]} ({', '.join(_EXERCISES[labels[0]])})"
    nums = [label.split()[-1] for label in labels]
    return "exercises " + ", ".join(nums[:-1]) + " and " + nums[-1]


def _table(headers: list[str], rows: list[list[str]]) -> None:
    """Print a plain-text table with right-aligned columns."""
    widths = [max(len(str(x)) for x in col) for col in zip(headers, *rows)]
    print("  ".join(f"{h:>{w}}" for h, w in zip(headers, widths)))
    for row in rows:
        print("  ".join(f"{str(c):>{w}}" for c, w in zip(row, widths)))


# --- COMMONS TIME LIMIT v4 --- every check in this notebook runs under a time limit
import contextlib, gc, os, signal, subprocess, sys, threading, time


# A time limit for every check that calls the student's code: a loop that never ends must fail
# its check, not freeze the notebook or the grader (which imports the notebook). Copy this block
# unchanged; change only the three limits below it. Why it is this careful:
#  * Whoever runs the notebook may already have an alarm running (tools/grade.py stops every
#    rubric row after 45 s; a harness may have one whose default action ends the process). This
#    guard saves it, lets whichever deadline comes first win (handing the signal to the outer
#    handler when it is the outer's turn) and on the way out re-arms it with the time it had left
#    and its interval. Switching it off instead would let the NEXT hang run for ever.
#  * Its own alarm is one-shot, re-armed by hand every 0.25 s once the time is up (in case a bare
#    `except:` swallows the first). A second alarm alone is not enough: `while True: try: ...
#    except: pass` is interrupted inside the `try` every time and carries on. So from the second
#    stop on, a line tracer (sys.settrace, on the running frames too) raises the stop again at the
#    first line the code runs afterwards, which is in the handler, outside the `try`; it is put
#    back as it was on the way out. A repeating alarm kept firing while a runaway loop's huge list
#    was being torn down; a repeat that arrived after the default handler was back ended the
#    process (exit -14, a dead Jupyter kernel), and one that arrived inside contextlib's exit
#    escaped as a raw timeout and left this handler installed.
#    On Python 3.12+ that tracer is sys.monitoring, which stays on after it raises, so a loop
#    inside TWO bare `except:` blocks is stopped as well: the Nth stop lets it raise N-1 times in
#    a row (one per handler it has to leave), and no more, so the `finally:` blocks and context
#    managers that run as the stop travels out are not interrupted again. On 3.11 sys.settrace is
#    unset by the interpreter each time it raises, so only the innermost one is peeled per alarm.
#  * It also watches memory: every 0.1 s it compares the process's resident memory with what it
#    was when the block started, and stops the block as soon as the growth passes `memory_mib`.
#    A loop that appends to a list reaches gigabytes in seconds, long before any sensible time
#    limit, so the time limit alone does not bound it. (Resident memory NOW, not the peak so
#    far: a peak only ever rises, so after one runaway check the next would be allowed another
#    `memory_mib` on top of it, and five looping exercises would reach ten gigabytes. Where the
#    system will not say, /proc on Linux and libproc on macOS being the two that do, it falls
#    back to the peak.)
#  * It never raises inside its own set-up, exit or clean-up (nor inside another guard's: guards
#    nest, the grader's wraps this one), and on the way out it takes any alarm of its own that has
#    gone off but not yet been handled (with SIGALRM blocked) before the old handler is put back.
# tools/verify_all.py fails a lesson whose guard does less.
class _OutOfTime(BaseException):
    """Raised inside the guarded block when its time is up. A BaseException, so `except
    Exception:` in the code under test cannot swallow it; it repeats every 0.25 s, and from the
    second time on a line tracer raises it again at the next line the code runs (inside the
    handler that swallowed the first), in case a bare `except:` does. `_time_limit` turns it into
    an AssertionError once it has left that code."""


def _resident() -> float:
    """This process's resident memory now, in MiB (Linux: /proc; macOS: libproc); its peak so far elsewhere."""
    try:
        if sys.platform.startswith("linux"):
            with open("/proc/self/statm") as f:
                return int(f.read().split()[1]) * os.sysconf("SC_PAGE_SIZE") / 2**20
        if sys.platform == "darwin":
            import ctypes, struct
            info = ctypes.create_string_buffer(96)           # struct proc_taskinfo
            if ctypes.CDLL("/usr/lib/libproc.dylib").proc_pidinfo(os.getpid(), 4, 0, info, 96) == 96:
                return struct.unpack_from("QQ", info.raw)[1] / 2**20     # pti_resident_size
    except Exception:
        pass
    import resource
    return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / (2**20 if sys.platform == "darwin" else 2**10)


@contextlib.contextmanager
def _time_limit(seconds: float, what: str, memory_mib: float = 0):
    """Fail the block with an AssertionError naming `what` if it runs longer than `seconds`, or
    (when `memory_mib` is set) if the process's resident memory grows by more than that while it runs.
    Uses the operating system's alarm (Linux and macOS, main thread); a no-op elsewhere."""
    if not (all(hasattr(signal, name) for name in ("setitimer", "SIGALRM", "pthread_sigmask", "sigpending", "sigwait"))
            and threading.current_thread() is threading.main_thread()):
        yield
        return
    if getattr(_time_limit, "stopped", False):           # a stopped check's frames keep what they built in a
        _time_limit.stopped = False                      # reference cycle until now: free it, so it is not counted
        gc.collect()                                     # as the growth of this block
    message = (f"{what} was still running after {seconds:g} s: a loop whose stopping condition "
               "is never met runs for ever; give every loop a cap")
    started = time.monotonic()
    used_mib = None
    if memory_mib:
        used_mib = _resident
        base_mib = used_mib()                            # resident memory now; only growth beyond it counts
    previous = signal.getsignal(signal.SIGALRM)
    outer_left, outer_every = signal.getitimer(signal.ITIMER_REAL)   # an alarm already running
    # One alarm at a time, never a repeating one. next: when ours is due; outer: when the alarm we
    # found is due (every: its interval); unseen: an alarm is armed and has not reached _fire yet.
    state = {"next": started + seconds, "outer": started + outer_left if outer_left > 0 else None,
             "every": outer_every, "unseen": False, "fired": False, "done": False, "message": message, "count": 0, "budget": 0,
             "traced": False, "tracer": None, "tool": None, "at": None, "thread": threading.get_ident(),
             "me": sys._getframe(), "owner": None}
    owner = state["me"].f_back
    while owner is not None and owner.f_globals is vars(contextlib):    # contextlib's __enter__
        owner = owner.f_back
    state["owner"], state["at"] = owner, owner.f_lineno   # the frame running the `with`, and its line

    def _arm(now, soonest=0.001):                        # one shot, at whichever deadline is next
        due = [t for t in (state["next"], state["outer"]) if t is not None]
        if used_mib and state["next"] is not None and not state["fired"]:
            due.append(now + 0.1)                        # and look at the memory every 0.1 s
        if due:
            signal.setitimer(signal.ITIMER_REAL, max(min(due) - now, soonest))
            state["unseen"] = True

    def _in_guard(frame):                                # is the alarm landing in a guard's own set-up, exit
        while frame is not None and frame is not state["owner"]:     # or clean-up (this one's or another's:
            code = frame.f_code                          # guards nest), or in contextlib's __exit__ under
            if (frame is state["me"] or code in _own_code or code.co_name in _GUARDS   # a `with`?
                    or frame.f_globals.get("__name__") == "contextlib"):
                return True
            frame = frame.f_back
        return False

    def _fire(signum, frame):
        state["unseen"] = False
        if state["done"]:
            return                                       # a late alarm of ours: already handled
        if _in_guard(frame):                             # never raise inside the guard: it is
            _arm(time.monotonic(), 0.01)                 # finishing; look again shortly
            return
        now = time.monotonic()
        if state["outer"] is not None and now >= state["outer"] - 0.005:
            every = state["every"]                       # the outer alarm is due: act as it would
            state["outer"] = state["outer"] + every * (1 + (now - state["outer"]) // every) if every else None
            try:
                if callable(previous):
                    previous(signum, frame)
                elif previous != signal.SIG_IGN:         # its default action ends the process
                    signal.signal(signal.SIGALRM, signal.SIG_DFL)
                    os.kill(os.getpid(), signal.SIGALRM)
            except BaseException:
                state["next"] = None                     # the outer one is stopping the block
                raise
            finally:
                left, every = signal.getitimer(signal.ITIMER_REAL)
                if left > 0:                             # the outer handler armed its next alarm
                    state["outer"], state["every"] = time.monotonic() + left, every
                _arm(time.monotonic())
            now = time.monotonic()
        grown = used_mib() - base_mib if used_mib and not state["fired"] else 0
        if grown > memory_mib and state["next"] is not None:       # out of memory before out of time
            state["message"] = (f"{what} was using over {memory_mib:g} MiB more memory than when it started, "
                                f"after {now - started:.1f} s: a loop that keeps adding to a list or a string "
                                "never stops; give every loop a cap")
            state["next"] = now
        if state["next"] is not None and now >= state["next"] - 0.005:
            swallowed = state["fired"]                   # a second stop: the first did not end the block
            state["fired"] = True
            state["count"] += 1
            state["next"] = now + 0.25                   # again, in case a bare `except:` swallows it
            _arm(now)
            if swallowed:
                state["budget"] = state["count"] - 1     # the tracer raises this many times, then leaves clean-up alone
                _trace_from(frame)
            raise _OutOfTime(state["message"], state)
        _arm(now)

    def _trace(frame, event, arg):                       # raises the stop at the next line of code that
        if state["done"] or _in_guard(frame):            # runs: inside the handler that swallowed the last
            return None                                  # (the interpreter then unsets it; the next alarm
        if event == "line":                              # sets it again)
            raise _OutOfTime(state["message"], state)
        return _trace

    def _line(code, line):                               # Python 3.12+: sys.monitoring, which is not unset
        if state["done"] or state["budget"] <= 0 or threading.get_ident() != state["thread"]:   # not unset when a
            return                                       # callback raises: the next line the code runs raises too, handlers
        frame = sys._getframe(1)                         # included (this thread only), but only `budget` times:
        if _in_guard(frame) or (frame is state["owner"] and line == state["at"]):   # clean-up that runs once the stop has
            return                                       # left the swallowing loops must not be interrupted again
        state["budget"] -= 1                             # (not the `with` line: its exit runs the guard)
        raise _OutOfTime(state["message"], state)

    def _trace_from(frame):                              # trace the running frames, and the ones called next
        if not state["traced"]:
            state["traced"] = True
            mon = getattr(sys, "monitoring", None)
            for tool in (5, 4, 3, 2, 1, 0) if mon is not None else ():
                try:
                    mon.use_tool_id(tool, "commons-time-limit")
                except ValueError:                       # a debugger or a profiler has this one
                    continue
                state["tool"] = tool
                mon.register_callback(tool, mon.events.LINE, _line)
                mon.set_events(tool, mon.events.LINE)
                return
            state["tracer"] = sys.gettrace()
            sys.settrace(_trace)
        while frame is not None and frame is not state["owner"]:
            frame.f_trace = _trace
            frame = frame.f_back

    _GUARDS = ("_time_limit", "_deadline", "_resident", "_fire", "_arm", "_trace", "_trace_from", "_line", "_in_guard")
    _own_code = (_fire.__code__, _arm.__code__, _trace.__code__, _trace_from.__code__, _line.__code__)
    signal.signal(signal.SIGALRM, _fire)
    _arm(time.monotonic())
    try:
        try:
            yield
        finally:
            state["done"] = True                         # from here on an alarm of ours only counts
            if state["traced"]:
                state["traced"] = False
                if state["tool"] is not None:
                    sys.monitoring.set_events(state["tool"], 0)
                    sys.monitoring.register_callback(state["tool"], sys.monitoring.events.LINE, None)
                    sys.monitoring.free_tool_id(state["tool"])
                else:
                    sys.settrace(state["tracer"])        # the tracer that was there before, if any
            mask = signal.pthread_sigmask(signal.SIG_BLOCK, {signal.SIGALRM})
            try:
                if signal.setitimer(signal.ITIMER_REAL, 0)[0] == 0 and state["unseen"]:
                    # Ours went off and has not reached _fire: it is pending here, or on its way to
                    # another thread. Take it before the old handler (by default, one that ends the
                    # process) is back.
                    give_up = time.monotonic() + 0.5
                    while state["unseen"] and time.monotonic() < give_up:
                        if signal.SIGALRM in signal.sigpending():
                            signal.sigwait({signal.SIGALRM})
                            state["unseen"] = False
                        else:
                            time.sleep(0.001)            # another thread has it: _fire runs next
                signal.signal(signal.SIGALRM, signal.SIG_DFL if previous is None else previous)
                outer = state["outer"]                   # re-arm the alarm we found, last
                signal.setitimer(signal.ITIMER_REAL, *((max(outer - time.monotonic(), 0.001), state["every"])
                                                       if outer is not None else (0, 0)))
            finally:
                signal.pthread_sigmask(signal.SIG_SETMASK, mask)
                state["me"] = state["owner"] = None
    except _OutOfTime as exc:
        if exc.args[-1] is not state:                    # an enclosing guard's: not ours to report
            raise
        _time_limit.stopped = True
        raise AssertionError(state["message"]) from None
    if state["fired"]:                                   # the block caught our alarm and carried on
        _time_limit.stopped = True
        raise AssertionError(state["message"])


# The three limits. A correct answer needs a small fraction of the first two (at least five times
# less, measured on the worked solution, on the slower of a Mac and an emulated Linux machine), so
# only a loop that never stops, or one that keeps appending until memory runs out, ever meets one.
_CHECK_SECONDS = 2     # an exercise's check
_DEMO_SECONDS = 120    # any other cell that runs your code: a demo, a table, a figure
_MEMORY_MIB = 2048     # how far a check may grow the process's resident memory before it is stopped


def _limited(label: str, check):
    """Run `check()` under `_time_limit` and return what it returns: _CHECK_SECONDS for an
    exercise's check, _DEMO_SECONDS for anything else. A check that is stopped (too long, or too
    much memory) fails with an AssertionError that says so; whatever the stopped code had built (a
    runaway list can hold gigabytes) is freed at once, not whenever the garbage collector next
    runs; and every program it had started that is still running is stopped too."""
    seconds = _CHECK_SECONDS if label.lower().startswith("exercise") else _DEMO_SECONDS
    before = _processes()
    try:
        with _time_limit(seconds, "this check", _MEMORY_MIB):
            return check()
    except AssertionError as exc:
        if not str(exc).startswith("this check was "):
            raise                                        # the check's own verdict, unchanged
        message = str(exc)
    gc.collect()          # the stopped frames hold what they built in a reference cycle until now
    _stop_strays(before)
    raise AssertionError(message)


def _processes() -> dict:
    """{pid: (parent, process group)} of every process (Linux: /proc; macOS: ps); {} if unreadable."""
    table = {}
    try:
        if os.path.isdir("/proc/self"):
            for entry in os.listdir("/proc"):
                if entry.isdigit():
                    try:
                        with open(f"/proc/{entry}/stat") as f:
                            fields = f.read().rsplit(")", 1)[1].split()
                        table[int(entry)] = (int(fields[1]), int(fields[2]))
                    except (OSError, ValueError, IndexError):
                        continue
        else:
            listing = subprocess.run(["ps", "-A", "-o", "pid=,ppid=,pgid="], capture_output=True, text=True,
                                     timeout=10).stdout
            for row in listing.splitlines():
                pid, ppid, pgid = (int(x) for x in row.split())
                table[pid] = (ppid, pgid)
    except (OSError, ValueError, subprocess.SubprocessError):
        return {}
    return table


def _stop_strays(before: dict) -> None:
    """Stopping a check stops the program it was waiting for, but not what THAT program started:
    stop `make test` and the test binary make launched runs on, at full speed, for ever. Stop every
    process that did not exist before the check (`before`, from _processes()) and either descends
    from this one or sits in its process group (an orphan: its parent is gone, so it now belongs to
    pid 1 or to a subreaper, whichever this machine has). Nothing that was already running is
    touched, the launcher that started this notebook included. Best effort."""
    now, me, group = _processes(), os.getpid(), os.getpgrp()
    if not before or not now:
        return
    def descends(pid):
        seen = set()
        while pid in now and pid not in seen:
            seen.add(pid)
            pid = now[pid][0]
            if pid == me:
                return True
        return False
    for pid, (_, pgid) in now.items():
        if pid != me and pid not in before and (pgid == group or descends(pid)):
            try:
                os.kill(pid, signal.SIGKILL)
            except OSError:
                pass
# --- END COMMONS TIME LIMIT ---


def _ready(*names: str) -> bool:
    """True when the earlier cells that define `names` have run; otherwise say so and let the caller skip."""
    missing = [name for name in names if name not in globals()]
    if missing:
        print("skipped — an earlier cell has not run (missing: " + ", ".join(missing) + "). Run the notebook top to bottom.")
    return not missing


def _try(label: str, check: Callable[[], None], needs: tuple[str, ...] = ()) -> None:
    """Run a check, or a demo that depends on your code, without derailing the notebook.

    A stub you have not filled in yet simply says so. A wrong answer prints the check's own
    message, which names the likely mistake, and the notebook carries on, so one broken
    exercise never hides the feedback on the others. A demo names the exercises it `needs`:
    until each has passed its check, the demo says which one it is waiting for and skips.
    Nothing is swallowed: every outcome is recorded in `_STATUS` for the progress board at the
    foot of the notebook, and every failure in `_FAILED_CHECKS`, which ends a script run
    non-zero.
    """
    waiting = [name for name in _EXERCISES
               if name in needs and _STATUS.get(name) != "passed"]
    if waiting:
        _STATUS[label] = "not started"
        print(f"{label}: skipped — needs {_named(waiting)} to pass first.")
        return
    try:
        _limited(label, check)
    except NotImplementedError as exc:
        _STATUS[label] = "not started"
        stub = traceback.extract_tb(exc.__traceback__)[-1].name
        owner = [name for name, funcs in _EXERCISES.items() if stub in funcs and name != label]
        if owner:
            print(f"{label}: skipped — needs {_named(owner)} first.")
        elif label in _EXERCISES:
            print(f"{label}: not implemented yet — fill in {stub}() above, then re-run "
                  "this cell.")
        else:
            print(f"{label}: skipped — {stub}() is not implemented yet.")
    except AssertionError as exc:
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: FAILED — {exc}")
    except Exception as exc:
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: raised {type(exc).__name__}: {exc}")
    else:
        _STATUS[label] = "passed"

## 1. The world: a prompt, a reference model, and a labeller with a hidden habit

A preference method needs three things: a **reference model** that writes answers, a set of
**preference pairs** (for one prompt, a chosen answer and a rejected one), and a way to tell how
good an answer really was. This lesson builds all three at a size where every number can be
computed in seconds.

* A **prompt** is four letters from `a`..`f`, for example `bdca`. The task is to answer with the
  prompt **reversed** (`acdb`), then optionally some `x` filler tokens, then an end token `.`.
* The **true quality** of an answer is the number of its first four letters that are right. Filler
  tokens are worth nothing and cost nothing: the truth does not care how long an answer is.
* The **reference model** is trained to imitate noisy answers: each letter is wrong with some
  probability, and the filler count is random. It is the "SFT model" that DPO starts from.
* The **labeller** compares two answers and prefers one by a noisy draw whose score is the true
  quality **plus a bonus per token of length**. That bonus is planted by this lesson, to stand for
  a known failure of real preference data (`claims.yaml` has the sources): raters, and the models
  trained on them, tend to prefer longer answers. Nothing in the pairs says so. Finding it, and
  measuring what a DPO-trained model does with it, is part of the lesson.

A sequence is laid out as `prompt | answer . ___`, padded to a fixed width. A **response mask**
marks the positions whose target is an answer token, because the loss counts only those (the lesson
of T04-L02): the model is not asked to predict the prompt. Run the cell to see the format.

In [ ]:
LETTER_NAMES = CHARS[:N_LETTERS]


def decode(tokens) -> str:
    """Token ids to a string, for printing."""
    return "".join(CHARS[int(t)] for t in tokens)


def all_prompts() -> np.ndarray:
    """Every possible prompt, in a seeded shuffled order: the first N_TRAIN_PROMPTS train, the rest are held out."""
    grid = np.stack(np.meshgrid(*[np.arange(N_LETTERS)] * PROMPT_LEN, indexing="ij"), axis=-1).reshape(-1, PROMPT_LEN)
    return grid[np.random.default_rng([SEED, 1]).permutation(len(grid))]


def sft_sequences(prompts: np.ndarray, rng) -> np.ndarray:
    """Noisy training answers for the reference model: the reversed prompt with each letter wrong with
    probability P_WRONG, then a random number of filler tokens (at most MAX_FILL), then the end token."""
    n = len(prompts)
    answer = prompts[:, ::-1].copy()
    wrong = rng.random((n, PROMPT_LEN)) < P_WRONG
    shift = rng.integers(1, N_LETTERS, size=(n, PROMPT_LEN))
    answer = np.where(wrong, (answer + shift) % N_LETTERS, answer)
    fillers = np.minimum(rng.geometric(0.5, size=n) - 1, MAX_FILL)
    tokens = np.full((n, WIDTH), PAD)
    tokens[:, :PROMPT_LEN] = prompts
    tokens[:, PROMPT_LEN] = SEP
    tokens[:, PROMPT_LEN + 1:PROMPT_LEN + 1 + PROMPT_LEN] = answer
    cols = np.arange(WIDTH)[None, :]
    first_fill = PROMPT_LEN + 1 + PROMPT_LEN
    tokens = np.where((cols >= first_fill) & (cols < first_fill + fillers[:, None]), FILL, tokens)
    tokens[np.arange(n), first_fill + fillers] = EOS
    return tokens


def response_mask(tokens: np.ndarray) -> np.ndarray:
    """(B, CTX) float mask over TARGET positions: 1.0 where the target token is part of the response.

    Position j of the model's output predicts token j + 1, so the targets are tokens[:, 1:]. The prompt
    and the separator are never targets of the loss; the padding after the end token is not either."""
    mask = (tokens[:, 1:] != PAD).astype(np.float64)
    mask[:, :PROMPT_LEN] = 0.0
    return mask


def response_features(tokens: np.ndarray, prompts: np.ndarray) -> dict:
    """What the TRUTH sees of each response: correct letters, length before the end token, fillers, and
    whether the response was cut off at MAX_RESP tokens without ending."""
    resp = tokens[:, PROMPT_LEN + 1:]
    is_end = resp == EOS
    ended = is_end.any(axis=1)
    length = np.where(ended, is_end.argmax(axis=1), MAX_RESP)
    inside = np.arange(MAX_RESP)[None, :] < length[:, None]
    return {"correct": (resp[:, :PROMPT_LEN] == prompts[:, ::-1]).sum(axis=1),
            "length": length,
            "fillers": ((resp == FILL) & inside).sum(axis=1),
            "capped": ~ended}


if __name__ == "__main__":
    ALL_PROMPTS = all_prompts()
    TRAIN_PROMPTS, HELD_PROMPTS = ALL_PROMPTS[:N_TRAIN_PROMPTS], ALL_PROMPTS[N_TRAIN_PROMPTS:]
    demo = sft_sequences(TRAIN_PROMPTS[:5], np.random.default_rng([SEED, 2]))
    print(f"{len(ALL_PROMPTS)} possible prompts: {len(TRAIN_PROMPTS)} to train on, {len(HELD_PROMPTS)} held out\n")
    print("five training sequences (prompt | answer . padding), the mask below each marks response targets:")
    for row, m in zip(demo, response_mask(demo)):
        print(f"  {decode(row)}   mask {''.join('1' if v else '0' for v in m)}")
    feats = response_features(demo, TRAIN_PROMPTS[:5])
    print("\ntrue quality of each (correct letters of the first four):", feats["correct"].tolist())
    print("length of each (tokens before the end token):             ", feats["length"].tolist())

## 2. The model, carried over from T03-L03

The model is the one you built in T03-L03, copied compactly so this notebook runs alone: token and
position embeddings, **one head of causal self-attention**, a residual connection, and a linear
layer to the vocabulary. No normalisation, no MLP. Two things differ from T03-L03, both on purpose:

* the **backward pass takes the gradient with respect to the logits as an argument**. In T03-L03
  it called cross-entropy's gradient itself. Here any loss can drive it, and DPO needs exactly
  that: the loss is a function of whole-sequence log-likelihoods, so what flows into the model is a
  different gradient on each position.
* the width is 32 and the vocabulary has ten symbols, so a training run takes seconds.

Everything in this cell is **given**. The backward pass is the hand derivation of T03-L03 (softmax
of masked scores, then the chain rule through values, scores, queries and keys, with the residual
path added). The next cell checks it against finite differences, so you can trust what you build on.
The training loop here uses ordinary cross-entropy (the gradient of the log-likelihood of the
tokens the mask marks), which is how the reference model is made; DPO's gradient will be a
re-weighting of exactly these terms.

In [ ]:
def _softmax(x: np.ndarray) -> np.ndarray:
    e = np.exp(x - x.max(axis=-1, keepdims=True))
    return e / e.sum(axis=-1, keepdims=True)


def init_params(seed: int = 1234, scale: float = 0.1) -> dict:
    rng = np.random.default_rng(seed)
    p = {"E": rng.normal(0, scale, (VOCAB, D_MODEL)), "P": rng.normal(0, scale, (CTX, D_MODEL))}
    for name in ("Wq", "Wk", "Wv", "Wo"):
        p[name] = rng.normal(0, scale, (D_MODEL, D_MODEL))
    p["W1"] = rng.normal(0, scale, (D_MODEL, VOCAB))
    p["b1"] = np.zeros(VOCAB)
    return p


def forward(params: dict, inputs: np.ndarray):
    """Logits (B, T, VOCAB) for every position, and the cache the backward pass needs."""
    t = inputs.shape[1]
    x = params["E"][inputs] + params["P"][None, :t]
    q, k, v = x @ params["Wq"], x @ params["Wk"], x @ params["Wv"]
    scores = (q @ k.transpose(0, 2, 1)) / np.sqrt(D_MODEL)
    scores = np.where(np.tril(np.ones((t, t), dtype=bool)), scores, -np.inf)   # causal: no looking ahead
    attn = _softmax(scores)
    context = attn @ v
    h = x + context @ params["Wo"]
    logits = h @ params["W1"] + params["b1"]
    return logits, {"inputs": inputs, "x": x, "q": q, "k": k, "v": v, "attn": attn, "context": context, "h": h}


def backward(params: dict, cache: dict, d_logits: np.ndarray) -> dict:
    """Gradients of a scalar loss with respect to every parameter, given the gradient `d_logits` of that
    loss with respect to the logits. The hand derivation of T03-L03, with the loss left open."""
    d = D_MODEL
    t = d_logits.shape[1]
    g = {"W1": cache["h"].reshape(-1, d).T @ d_logits.reshape(-1, VOCAB), "b1": d_logits.sum(axis=(0, 1))}
    d_h = d_logits @ params["W1"].T
    g["Wo"] = cache["context"].reshape(-1, d).T @ d_h.reshape(-1, d)
    d_context = d_h @ params["Wo"].T
    attn, q, k, v, x = cache["attn"], cache["q"], cache["k"], cache["v"], cache["x"]
    d_attn = d_context @ v.transpose(0, 2, 1)
    d_v = attn.transpose(0, 2, 1) @ d_context
    d_scores = attn * (d_attn - (d_attn * attn).sum(axis=-1, keepdims=True)) / np.sqrt(d)
    d_q, d_k = d_scores @ k, d_scores.transpose(0, 2, 1) @ q
    g["Wq"] = x.reshape(-1, d).T @ d_q.reshape(-1, d)
    g["Wk"] = x.reshape(-1, d).T @ d_k.reshape(-1, d)
    g["Wv"] = x.reshape(-1, d).T @ d_v.reshape(-1, d)
    d_x = d_h + d_q @ params["Wq"].T + d_k @ params["Wk"].T + d_v @ params["Wv"].T
    g["P"] = np.zeros_like(params["P"])
    g["P"][:t] = d_x.sum(axis=0)
    g["E"] = np.zeros_like(params["E"])
    np.add.at(g["E"], cache["inputs"], d_x)      # a token can occur many times: its gradients add
    return g


def adam_init(params: dict) -> dict:
    return {k: [np.zeros_like(v), np.zeros_like(v)] for k, v in params.items()}


def adam_step(params: dict, grads: dict, state: dict, step: int, lr: float) -> None:
    """One Adam update, in place."""
    for k, p in params.items():
        m, v = state[k]
        m *= 0.9
        m += 0.1 * grads[k]
        v *= 0.999
        v += 0.001 * grads[k] ** 2
        p -= lr * (m / (1 - 0.9 ** step)) / (np.sqrt(v / (1 - 0.999 ** step)) + 1e-8)


def nll_loss_and_dlogits(logits: np.ndarray, targets: np.ndarray, mask: np.ndarray):
    """Mean negative log-likelihood per masked token, and its gradient with respect to the logits. Given."""
    probs = _softmax(logits)
    picked = np.take_along_axis(probs, targets[..., None], axis=-1)[..., 0]
    count = mask.sum()
    loss = float(-(np.log(picked) * mask).sum() / count)
    one_hot = np.zeros_like(probs)
    np.put_along_axis(one_hot, targets[..., None], 1.0, axis=-1)
    return loss, (probs - one_hot) * mask[..., None] / count


def fit_nll(params: dict, draw_batch: Callable, steps: int, lr: float, seed: int, report: int = 0) -> list:
    """Train `params` in place by maximum likelihood of the masked tokens of `draw_batch(rng)`. Returns the losses."""
    rng = np.random.default_rng([SEED, seed])
    state, losses = adam_init(params), []
    for step in range(1, steps + 1):
        tokens = draw_batch(rng)
        logits, cache = forward(params, tokens[:, :-1])
        loss, d_logits = nll_loss_and_dlogits(logits, tokens[:, 1:], response_mask(tokens))
        adam_step(params, backward(params, cache, d_logits), state, step, lr)
        losses.append(loss)
        if report and (step == 1 or step % report == 0):
            print(f"  step {step:5d}  loss {loss:6.3f} nats per answer token")
    return losses


def _verify_backward() -> None:
    """The model's backward pass against central finite differences, at random coordinates of every parameter."""
    rng = np.random.default_rng([SEED, 3])
    params = init_params(seed=5, scale=0.3)
    tokens = sft_sequences(all_prompts()[:3], rng)
    inputs, targets, mask = tokens[:, :-1], tokens[:, 1:], response_mask(tokens)
    weights = rng.normal(size=(3, CTX, VOCAB))          # an arbitrary loss: a random linear function of the logits

    def loss() -> float:
        return float((forward(params, inputs)[0] * weights).sum())

    logits, cache = forward(params, inputs)
    grads = backward(params, cache, weights)
    worst = 0.0
    for name, array in params.items():
        for q in rng.integers(0, array.size, size=4):
            old = array.flat[q]
            array.flat[q] = old + 1e-5
            up = loss()
            array.flat[q] = old - 1e-5
            down = loss()
            array.flat[q] = old
            numeric, analytic = (up - down) / 2e-5, grads[name].flat[q]
            worst = max(worst, abs(numeric - analytic) / (1e-8 + 1e-4 * max(abs(numeric), abs(analytic))))
    print("backward pass against finite differences, 4 coordinates of each of the", len(params),
          "parameter arrays, for an arbitrary loss on the logits:")
    print("  every one within the tolerance:", worst <= 1.0)
    assert worst <= 1.0, "the given backward pass disagrees with finite differences; this is not an exercise, please report it"


if __name__ == "__main__":
    _try("backward pass check", _verify_backward)

## 3. The reference model, and what it writes

The reference is the model DPO will start from and be held close to. It is trained by maximum
likelihood on the noisy answers of section 1, which is **supervised fine-tuning (SFT)** in miniature.
After training it writes answers by **sampling**: at each position draw the next token from the
model's own distribution, until it draws the end token or reaches the length cap. The sampler never
draws the separator or the padding symbol (the two structural symbols are masked out of the draw);
the cell prints how much probability the trained model puts on them, so you can judge how much
this changes.

The cell trains the reference and then measures, on held-out prompts, the three things the rest of
the lesson tracks: **true quality** (correct letters of the first four), **length** (tokens before
the end token), and the share of answers **cut off** at the cap without ending.

In [ ]:
def sample_responses(params: dict, prompts: np.ndarray, rng) -> np.ndarray:
    """Sample one response per prompt from the model, token by token. Returns full padded sequences."""
    n = len(prompts)
    tokens = np.full((n, WIDTH), PAD)
    tokens[:, :PROMPT_LEN] = prompts
    tokens[:, PROMPT_LEN] = SEP
    done = np.zeros(n, dtype=bool)
    for pos in range(PROMPT_LEN + 1, WIDTH):
        logits, _ = forward(params, tokens[:, :pos])
        probs = _softmax(logits[:, -1])
        probs[:, (SEP, PAD)] = 0.0                                  # the structural symbols are never drawn
        cdf = np.cumsum(probs / probs.sum(axis=1, keepdims=True), axis=1)
        drawn = np.minimum((cdf < rng.random((n, 1))).sum(axis=1), VOCAB - 1)
        tokens[:, pos] = np.where(done, PAD, drawn)
        done |= drawn == EOS
    return tokens


def evaluate_policy(params: dict, seed: int = 50, repeats: int = 2) -> dict:
    """Sample `repeats` responses for every held-out prompt and report what the truth sees of them."""
    prompts = np.repeat(HELD_PROMPTS, repeats, axis=0)
    tokens = sample_responses(params, prompts, np.random.default_rng([SEED, seed]))
    f = response_features(tokens, prompts)
    return {"quality": float(f["correct"].mean()), "length": float(f["length"].mean()),
            "fillers": float(f["fillers"].mean()), "capped": float(f["capped"].mean()), "tokens": tokens, "prompts": prompts}


def structural_mass(params: dict) -> float:
    """The largest probability the model puts on the separator or padding symbol at a response position,
    over 200 reference-style sequences."""
    tokens = sft_sequences(TRAIN_PROMPTS[:200], np.random.default_rng([SEED, 4]))
    probs = _softmax(forward(params, tokens[:, :-1])[0])
    return float(probs[:, PROMPT_LEN:][..., (SEP, PAD)].max())


if __name__ == "__main__":
    print("training the reference model on noisy answers (supervised fine-tuning):")
    REF = init_params()
    fit_nll(REF, lambda rng: sft_sequences(TRAIN_PROMPTS[rng.integers(0, N_TRAIN_PROMPTS, size=64)], rng),
            REF_STEPS, REF_LR, seed=5, report=500)
    ref_eval = evaluate_policy(REF)
    print(f"\nthe model has {sum(v.size for v in REF.values())} parameters, width {D_MODEL}, and a vocabulary of {VOCAB} symbols")
    print(f"reference, sampled on {len(ref_eval['prompts'])} held-out prompts:")
    print(f"  true quality   {ref_eval['quality']:.2f} correct letters of {PROMPT_LEN}")
    print(f"  length         {ref_eval['length']:.2f} tokens before the end token ({ref_eval['fillers']:.2f} of them filler)")
    print(f"  cut off        {100 * ref_eval['capped']:.1f}% of answers reached the {MAX_RESP}-token cap")
    print(f"  the largest probability on a structural symbol at a response position: {structural_mass(REF):.4f}")

## 4. Preference pairs, and an audit of the labeller

DPO learns from pairs. The recipe the DPO paper describes is: for each prompt, sample two answers
from the reference model, ask a labeller which is better, and keep the pair as (chosen, rejected).
The cell does that for the training prompts (several pairs per prompt, with pairs whose two answers
are identical dropped, since they say nothing) and, separately, for the held-out prompts, so that
the margin of a trained model can be measured on pairs it never saw.

The labeller is the one of section 1: it draws the winner of a pair with probability
`sigmoid(score_1 - score_2)`, where an answer's score is its true quality plus a bonus per token of
length. That is the Bradley-Terry model the DPO derivation also assumes, with the planted bonus in
the score.

## Exercise 1: `win_counts()`

Before training on a set of pairs, **audit it** against what you know. `win_counts(score_w, score_l)`
counts, over pairs, how often the chosen answer's score is higher, lower and equal. With the optional
`group_w` and `group_l` it counts only the pairs in which the two answers share the same group value.
Used with *length* as the group and *true quality* as the score, it asks a question that can hint at a
length bias: **among pairs where both answers are equally long, how often does the labeller agree with
the truth, compared with the pairs whose lengths differ?** Used with length as the score, it counts how
often the chosen answer is the longer one. The cell after the exercise adds a control to read both against.
Return three plain Python ints: (wins, losses, ties). A tie means exactly equal.

<details><summary>💡 Hint 1 — what to think about</summary>

Three counts that sum to the number of pairs counted. Which pairs are counted when groups are
given? Is a tie a win for either side, and what does "equal" mean for integers?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Turn the scores into arrays. If groups are given, build a boolean selector that is true where the
two groups are equal and keep only those pairs. Then three comparisons, summed, each converted to
an int. Raise ValueError if the lengths differ or if only one of the two groups is given.
</details>

In [ ]:
def win_counts(score_w, score_l, group_w=None, group_l=None):
    """Count, over pairs, the chosen answer's score being higher, lower, or equal.

    `score_w[i]` and `score_l[i]` are the scores of the chosen and the rejected answer of pair i.
    If `group_w` and `group_l` are both given, only the pairs with `group_w[i] == group_l[i]` count
    (groups are compared with `==` and nothing else; scores need only support `>`, `<` and `==`).
    The inputs may be lists, tuples or arrays.
    Raise ValueError if the arrays differ in length, or if exactly one of the two groups is given.

    Example (three pairs; only the first and third share a group):
        >>> win_counts([3, 1, 2], [2, 1, 5])
        (1, 1, 1)
        >>> win_counts([3, 1, 2], [2, 1, 5], [7, 8, 9], [7, 6, 9])
        (1, 1, 0)

    Returns:
        (wins, losses, ties) as three Python ints, wins meaning score_w > score_l.
    """
    # YOUR CODE HERE
    raise NotImplementedError("implement win_counts")


def _check_win_counts() -> None:
    got = win_counts([3, 1, 2], [2, 1, 5])
    assert got == (1, 1, 1), f"win_counts([3, 1, 2], [2, 1, 5]) gave {got}, expected (1, 1, 1): wins are score_w > score_l, losses score_w < score_l, ties exactly equal"
    got = win_counts([3, 1, 2], [2, 1, 5], [7, 8, 9], [7, 6, 9])
    assert got == (1, 1, 0), f"with groups [7, 8, 9] and [7, 6, 9] only pairs 0 and 2 count (one win, one loss); got {got}, expected (1, 1, 0)"
    assert all(type(x) is int for x in got), f"return plain Python ints, not {[type(x).__name__ for x in got]}"
    assert win_counts(np.array([3, 1, 2]), np.array([2, 1, 5]), np.array([7, 8, 9]), np.array([7, 6, 9])) == (1, 1, 0), "arrays work like lists"
    assert win_counts([], []) == (0, 0, 0), "no pairs should give (0, 0, 0)"
    for bad in (lambda: win_counts([1, 2], [1]), lambda: win_counts([1], [1], [1], None), lambda: win_counts([1], [1], None, [1])):
        try:
            bad()
        except ValueError:
            continue
        raise AssertionError("mismatched lengths, or only one of the two groups, should raise ValueError")
    print("exercise 1 looks right")


if __name__ == "__main__":
    _try("exercise 1", _check_win_counts)

In [ ]:
def preference_labeller(first: np.ndarray, second: np.ndarray, prompts: np.ndarray, rng, bonus: float = LENGTH_BONUS) -> np.ndarray:
    """True where the labeller prefers the first answer: a Bradley-Terry draw on (true quality + bonus * length)."""
    f1, f2 = response_features(first, prompts), response_features(second, prompts)
    score1 = f1["correct"] + bonus * f1["length"]
    score2 = f2["correct"] + bonus * f2["length"]
    return rng.random(len(prompts)) < 1.0 / (1.0 + np.exp(score2 - score1))


def build_pairs(params: dict, prompts: np.ndarray, per_prompt: int, seed: int) -> dict:
    """Sample two answers per (repeated) prompt from `params`, label them, drop identical pairs."""
    rng = np.random.default_rng([SEED, seed])
    rep = np.repeat(prompts, per_prompt, axis=0)
    first, second = sample_responses(params, rep, rng), sample_responses(params, rep, rng)
    first_wins = preference_labeller(first, second, rep, rng)
    keep = (first != second).any(axis=1)
    chosen = np.where(first_wins[:, None], first, second)[keep]
    rejected = np.where(first_wins[:, None], second, first)[keep]
    prompts_kept = rep[keep]
    fc, fr = response_features(chosen, prompts_kept), response_features(rejected, prompts_kept)
    return {"prompts": prompts_kept, "chosen": chosen, "rejected": rejected, "first": first[keep], "second": second[keep],
            "dropped": int((~keep).sum()), "total": len(rep),
            "q_w": fc["correct"], "q_l": fr["correct"], "len_w": fc["length"], "len_l": fr["length"]}


def _agreement(q_w, q_l, len_w, len_l):
    """Shares (with standard errors) of: agreement with the truth among pairs whose correctness differs, for equal
    lengths and for differing lengths (uses your exercise 1), and the chosen answer being the longer one."""
    quality_all, quality_same = win_counts(q_w, q_l), win_counts(q_w, q_l, len_w, len_l)
    quality_diff = tuple(a - b for a, b in zip(quality_all, quality_same))            # pairs whose lengths differ
    length_all = win_counts(len_w, len_l)
    share = lambda w, n: (w / n, float(np.sqrt(w / n * (1 - w / n) / n)))
    return {"counts": (quality_all, quality_same, quality_diff),
            "same": share(quality_same[0], quality_same[0] + quality_same[1]), "n_same": quality_same[0] + quality_same[1],
            "diff": share(quality_diff[0], quality_diff[0] + quality_diff[1]), "n_diff": quality_diff[0] + quality_diff[1],
            "longer": share(length_all[0], length_all[0] + length_all[1]), "n_longer": length_all[0] + length_all[1]}


def _gap(a: dict):
    """Equal-length agreement minus differing-length agreement, and that difference in standard errors."""
    gap = a["same"][0] - a["diff"][0]
    return gap, gap / float(np.hypot(a["same"][1], a["diff"][1]))


def _audit_labels() -> None:
    if not _ready("TRAIN_PAIRS"):
        return
    p = TRAIN_PAIRS
    print(f"{p['total']} pairs sampled from the reference, {p['dropped']} identical pairs dropped, {len(p['chosen'])} kept\n")
    planted = _agreement(p["q_w"], p["q_l"], p["len_w"], p["len_l"])
    rows = [["all pairs", *planted["counts"][0]], ["lengths equal", *planted["counts"][1]], ["lengths differ", *planted["counts"][2]]]
    print("is the chosen answer the more correct one? (true quality of the chosen against the rejected answer)")
    _table(["pairs counted", "chosen more correct", "less correct", "equally correct"], rows)
    print()
    for name, key in (("equal lengths", "same"), ("differing lengths", "diff")):
        print(f"  among pairs with {name} that differ in correctness ({planted['n_' + key]} pairs), the labeller agrees with the truth in {100 * planted[key][0]:.1f}% ± {100 * planted[key][1]:.1f} (one standard error)")
    gap, gap_z = _gap(planted)
    print(f"  equal lengths minus differing lengths: {100 * gap:+.1f} points, {gap_z:.1f} standard errors")
    print(f"  the chosen answer is the longer one in {100 * planted['longer'][0]:.1f}% ± {100 * planted['longer'][1]:.1f} of the {planted['n_longer']} pairs whose lengths differ")
    # The control: the SAME answer pairs, labelled again by a labeller with no length bonus.
    rng = np.random.default_rng([SEED, 8])
    first_wins = preference_labeller(p["first"], p["second"], p["prompts"], rng, bonus=0.0)
    chosen, rejected = np.where(first_wins[:, None], p["first"], p["second"]), np.where(first_wins[:, None], p["second"], p["first"])
    fc, fr = response_features(chosen, p["prompts"]), response_features(rejected, p["prompts"])
    blind = _agreement(fc["correct"], fr["correct"], fc["length"], fr["length"])
    blind_gap, blind_z = _gap(blind)
    print("\ncontrol: the same answer pairs, labelled again by a labeller that ignores length (same Bradley-Terry draw, bonus 0):")
    print(f"  equal lengths minus differing lengths: {100 * blind_gap:+.1f} points, {blind_z:.1f} standard errors")
    print(f"  the chosen answer is the longer one in {100 * blind['longer'][0]:.1f}% ± {100 * blind['longer'][1]:.1f} of the {blind['n_longer']} pairs whose lengths differ")
    excess = planted['longer'][0] - blind['longer'][0]
    print(f"\nso the planted labeller picks the longer answer {100 * abs(excess):.1f} points {'more' if excess > 0 else 'less'} often than a length-blind one does on these very answers."
          f" The equal-minus-differing contrast is the weaker test: even a length-blind labeller can show a gap there (here"
          f" {100 * blind_gap:+.1f} points, from one draw), because pairs of equal length and pairs of different length can differ in how far apart their answers are in correctness,"
          " and agreement with the truth depends on that. Lean on the share of chosen answers that are longer, set against the control.")

if __name__ == "__main__":
    TRAIN_PAIRS = build_pairs(REF, TRAIN_PROMPTS, PAIRS_PER_PROMPT, seed=6)
    HELD_PAIRS = build_pairs(REF, HELD_PROMPTS, 2, seed=7)
    _try("pair audit", _audit_labels, needs=("exercise 1",))

## 5. The log-likelihood of a response

DPO never asks the model for a sampled answer during training. It asks for one number per stored
answer: **how likely does the current model find this exact response, given this prompt?** For a
language model that probability is a product over tokens, so its logarithm is a **sum** of
per-token log-probabilities:

`log pi(y | x) = sum over response positions t of log softmax(logits[t])[y_t]`

## Exercise 2: `sequence_logprob()`

Given logits `(B, T, V)`, target token ids `(B, T)` and the response mask `(B, T)`, return the
log-likelihood of each sequence's masked tokens, shape `(B,)`. Two things matter. **Sum**, not mean:
a longer response is a smaller probability, and DPO's loss is built on the full sequence's ratio. And
**stability**: the log-softmax must survive logits of size ten thousand, so subtract the largest
logit before exponentiating and take the logarithm of the sum yourself. The positions the mask
excludes must contribute nothing, however large their logits.

<details><summary>💡 Hint 1 — what to think about</summary>

What is log softmax written as a difference of two terms? Which of the two can overflow, and what
can you subtract from every logit to prevent that without changing the answer?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Shift the logits by their row maximum, compute the log of the row sum of exponentials, and subtract
it to get log-probabilities for every symbol. Pick the target symbol's entry at every position, multiply
by the mask, and add up along the position axis only.
</details>

In [ ]:
def sequence_logprob(logits, targets, mask):
    """Log-likelihood of each sequence's masked target tokens: the sum of their log-softmax entries.

    Args:
        logits: (B, T, V) raw scores; may be huge (1e4) and must not overflow.
        targets: (B, T) integer token ids, the token each position should predict.
        mask: (B, T) of 0/1 (or bool): only positions with a 1 are summed.
    Each input may be an array or a nested list; convert with `np.asarray` first.

    Example (uniform scores over four symbols: every token has probability 1/4; two counted tokens):
        >>> sequence_logprob(np.zeros((1, 2, 4)), np.array([[0, 1]]), np.array([[1.0, 1.0]]))
        array([-2.77258872])

    Returns:
        a float array of shape (B,); the inputs are not modified.
    """
    # YOUR CODE HERE
    raise NotImplementedError("implement sequence_logprob")


def _check_sequence_logprob() -> None:
    got = sequence_logprob(np.zeros((1, 2, 4)), np.array([[0, 1]]), np.array([[1.0, 1.0]]))
    assert got.shape == (1,) and abs(got[0] - 2 * np.log(0.25)) < 1e-12, (
        f"uniform scores over four symbols should give 2 * log(1/4) = {2 * np.log(0.25):.6f}; got {got.tolist()}. Sum the log-probabilities of the masked tokens")
    got = sequence_logprob(np.zeros((1, 3, 4)), np.array([[0, 1, 2]]), np.array([[1.0, 0.0, 1.0]]))
    assert abs(got[0] - 2 * np.log(0.25)) < 1e-12, f"a masked-out position must add nothing; got {got.tolist()}"
    big = np.array([[[10000.0, 0.0, 0.0], [0.0, 10000.0, 0.0]]])
    with np.errstate(over="raise", invalid="raise", divide="raise"):
        got = sequence_logprob(big, np.array([[0, 1]]), np.array([[1.0, 1.0]]))
    assert np.isfinite(got).all() and abs(got[0]) < 1e-9, (
        f"logits of 10000 gave {got.tolist()}; subtract the largest logit of each row before exponentiating, so nothing overflows")
    rng = np.random.default_rng(0)
    z = rng.normal(size=(3, 4, 5))
    t = rng.integers(0, 5, size=(3, 4))
    m = rng.integers(0, 2, size=(3, 4))
    want = np.array([sum(np.log(np.exp(z[b, i, t[b, i]]) / np.exp(z[b, i]).sum()) for i in range(4) if m[b, i]) for b in range(3)])
    got = sequence_logprob(z, t, m)
    assert got.shape == (3,) and np.allclose(got, want, rtol=0, atol=1e-12), (
        f"a random batch gave {np.round(got, 4).tolist()}, expected {np.round(want, 4).tolist()}: sum over positions (axis -1), not over the batch, and not a mean")
    before = z.copy()
    sequence_logprob(z, t, m)
    assert np.array_equal(z, before), "your function changed the logits it was given (shifting them in place is the usual way); build a new array instead"
    got = sequence_logprob(z.tolist(), t.tolist(), m.tolist())
    assert np.allclose(got, want, rtol=0, atol=1e-12), "nested Python lists should work too: convert the inputs with np.asarray first"
    print("exercise 2 looks right")


if __name__ == "__main__":
    _try("exercise 2", _check_sequence_logprob)

In [ ]:
def reference_logprobs(ref_params: dict, pairs: dict):
    """The reference model's log-likelihood of each pair's chosen and rejected answer (uses your sequence_logprob)."""
    out = []
    for key in ("chosen", "rejected"):
        tokens = pairs[key]
        logits, _ = forward(ref_params, tokens[:, :-1])
        out.append(sequence_logprob(logits, tokens[:, 1:], response_mask(tokens)))
    return out[0], out[1]


def _show_logprobs() -> None:
    global REF_W, REF_L, HELD_REF_W, HELD_REF_L
    if not _ready("REF", "TRAIN_PAIRS", "HELD_PAIRS"):
        return
    REF_W, REF_L = reference_logprobs(REF, TRAIN_PAIRS)
    HELD_REF_W, HELD_REF_L = reference_logprobs(REF, HELD_PAIRS)
    n_resp = response_mask(TRAIN_PAIRS["chosen"]).sum(axis=1)
    print("the reference model's log-likelihood of the first five pairs' answers (nats):")
    rows = [[i, decode(TRAIN_PAIRS["chosen"][i, PROMPT_LEN + 1:]).rstrip("_"), f"{REF_W[i]:.2f}",
             decode(TRAIN_PAIRS["rejected"][i, PROMPT_LEN + 1:]).rstrip("_"), f"{REF_L[i]:.2f}"] for i in range(5)]
    _table(["pair", "chosen", "log pi_ref", "rejected", "log pi_ref"], rows)
    print(f"\nmean over all pairs: chosen {REF_W.mean():.2f}, rejected {REF_L.mean():.2f} nats; "
          f"per response token: chosen {(REF_W / n_resp).mean():.3f}, rejected "
          f"{(REF_L / response_mask(TRAIN_PAIRS['rejected']).sum(axis=1)).mean():.3f}")
    print("a sum over a longer response is a more negative number: the sum is the log of one probability, not an average.")


if __name__ == "__main__":
    _try("reference log-likelihoods", _show_logprobs, needs=("exercise 2",))

## 6. Deriving the DPO loss

**The RLHF objective.** For a prompt `x`, a reward `r(x, y)` and a reference model `pi_ref`, the
policy `pi` should earn reward without wandering far from the reference:

`J(pi) = E over y ~ pi of r(x, y)  -  beta * KL(pi || pi_ref)`

**Its optimum, in closed form.** Let `Z(x) = sum over y of pi_ref(y|x) * exp(r(x,y) / beta)` and
`pi*(y|x) = pi_ref(y|x) * exp(r(x,y) / beta) / Z(x)`. Expand `KL(pi || pi*)` and use
`log pi* = log pi_ref + r / beta - log Z`:

`J(pi) = beta * log Z(x)  -  beta * KL(pi || pi*)`

A KL divergence is never negative, so `J` is largest exactly when `pi = pi*`.

**Invert it.** Take logs of `pi*`: `r(x, y) = beta * log( pi*(y|x) / pi_ref(y|x) ) + beta * log Z(x)`.
The reward is the policy's log-ratio to the reference, scaled by `beta`, plus a term that depends only
on the prompt.

**Cancel what you cannot compute.** The Bradley-Terry model says the chance that `y_w` beats `y_l` is
`sigmoid( r(x, y_w) - r(x, y_l) )`. It depends only on a *difference* of rewards, and `Z(x)` is the same for
both answers, so it cancels. Replace `pi*` by the policy `pi_theta` you are training and take the
negative log-likelihood of the labels:

`L = - mean log sigmoid( beta * [ log pi(y_w|x)/pi_ref(y_w|x)  -  log pi(y_l|x)/pi_ref(y_l|x) ] )`

The quantity `beta * log( pi(y|x) / pi_ref(y|x) )` is the **implicit reward** of an answer, and its
difference between the chosen and the rejected answer is the **margin** `m`. The cell below checks
the two claims that the derivation leans on, on a toy with six possible answers.

In [ ]:
def _verify_derivation() -> None:
    rng = np.random.default_rng([SEED, 8])
    n, beta = 6, 0.5
    r = rng.normal(size=n)
    pi_ref = rng.dirichlet(np.ones(n))
    z = (pi_ref * np.exp(r / beta)).sum()
    pi_star = pi_ref * np.exp(r / beta) / z

    def objective(pi):
        return float((pi * r).sum() - beta * (pi * np.log(pi / pi_ref)).sum())

    def kl_to_star(pi):
        return float((pi * np.log(pi / pi_star)).sum())

    others = rng.dirichlet(np.ones(n), size=2000)
    identity = max(abs(objective(p) - (beta * np.log(z) - beta * kl_to_star(p))) for p in others)
    best_other = max(objective(p) for p in others)
    implicit = beta * np.log(pi_star / pi_ref)
    differences_ok = all(abs((implicit[i] - implicit[j]) - (r[i] - r[j])) < 1e-9 for i in range(n) for j in range(n))
    print("a toy with six possible answers, a random reward and a random reference policy:")
    print("  J(pi) = beta*log Z - beta*KL(pi||pi*) for 2000 random policies, to within 1e-9:", identity < 1e-9)
    print("  pi* beats all 2000 of them on J:", objective(pi_star) > best_other)
    print("  the implicit reward beta*log(pi*/pi_ref) differs from r by a constant (so every pair's difference matches), to within 1e-9:", differences_ok)
    assert identity < 1e-9 and objective(pi_star) > best_other and differences_ok, "the derivation check failed; this is not an exercise, please report it"


if __name__ == "__main__":
    _try("derivation check", _verify_derivation)

## 7. Exercise 3: `dpo_loss()`

Now the loss itself. You are given the policy's and the reference's log-likelihoods of the chosen
(`lp_w`, `ref_w`) and rejected (`lp_l`, `ref_l`) answers, one value per pair, and `beta`.

* the **margin** of each pair is `beta * ((lp_w - ref_w) - (lp_l - ref_l))`: the implicit reward of
  the chosen answer minus that of the rejected one;
* the **loss** is the mean over pairs of `-log sigmoid(margin)`.

At the start of training the policy *is* the reference, so every margin is zero. Write `-log sigmoid(m)`
so that it survives a margin of plus or minus a thousand without overflow: `log(1 + exp(-m))` overflows
for a very negative `m`, so find a safe way to compute the same quantity (hint 2 names the tool). Return the
mean loss as a Python float and the margins as an array. The inputs may be lists, tuples or arrays. Raise
`ValueError` if `beta` is not positive, if any of the four arrays differs in length from another, or if there are no pairs.

<details><summary>💡 Hint 1 — what to think about</summary>

Which four numbers go into the margin, and in what order of subtraction? Is the chosen answer's
term added or subtracted? What does `-log(sigmoid(m))` look like for a margin of -1000?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Form each answer's change from the reference (policy minus reference), subtract the rejected's from
the chosen's, multiply by beta. For the loss, `np.logaddexp(a, b)` evaluates `log(exp(a) + exp(b))`
without overflowing: write `log(1 + exp(-m))` as such a sum and apply it with the right signs.
</details>

In [ ]:
def dpo_loss(lp_w, lp_l, ref_w, ref_l, beta):
    """The DPO loss and the implicit reward margin of each pair.

    margin_i = beta * ((lp_w_i - ref_w_i) - (lp_l_i - ref_l_i)),   loss = mean_i of -log sigmoid(margin_i).
    Raise ValueError if beta <= 0, if any of the four arrays differs in length from another, or if there are no pairs.
    The four inputs may be lists, tuples or arrays; convert them with `np.asarray` first.

    Example (the policy likes the chosen answer more than the reference does, the rejected one less):
        >>> loss, margins = dpo_loss(np.array([-1.0]), np.array([-3.0]), np.array([-2.0]), np.array([-2.0]), 0.5)
        >>> margins
        array([1.])
        >>> round(loss, 4)
        0.3133

    Returns:
        (loss, margins): a Python float and a float array with one margin per pair.
    """
    # YOUR CODE HERE
    raise NotImplementedError("implement dpo_loss")


def _check_dpo_loss() -> None:
    loss, margins = dpo_loss([-1.0], [-3.0], [-2.0], [-2.0], 0.5)
    assert np.allclose(margins, [1.0], rtol=0, atol=1e-12), (
        f"margin should be 0.5 * ((-1 - -2) - (-3 - -2)) = 1.0; got {np.asarray(margins).tolist()}: subtract the reference from each answer first, then the rejected from the chosen")
    assert abs(loss - np.log1p(np.exp(-1.0))) < 1e-12, f"loss should be -log sigmoid(1) = {np.log1p(np.exp(-1.0)):.6f}; got {loss}"
    loss, margins = dpo_loss(np.array([-2.0, -5.0]), np.array([-2.0, -1.0]), np.array([-2.0, -5.0]), np.array([-2.0, -1.0]), 0.1)
    assert abs(loss - np.log(2.0)) < 1e-12 and np.allclose(margins, 0.0), (
        f"when the policy equals the reference every margin is 0 and the loss is log 2 = {np.log(2.0):.6f}; got loss {loss}, margins {np.asarray(margins).tolist()}")
    with np.errstate(over="raise", invalid="raise", divide="raise"):
        loss, margins = dpo_loss(np.array([-1000.0]), np.array([0.0]), np.array([0.0]), np.array([0.0]), 1.0)
    assert abs(loss - 1000.0) < 1e-9, f"a margin of -1000 should cost about 1000; got {loss}. Use np.logaddexp(0, -margin) so that exp() never overflows"
    loss, _ = dpo_loss(np.array([1000.0]), np.array([0.0]), np.array([0.0]), np.array([0.0]), 1.0)
    assert 0.0 <= loss < 1e-12, f"a margin of +1000 should cost about 0; got {loss}"
    assert type(loss) is float, f"return the loss as a Python float, not {type(loss).__name__}"
    for bad in (lambda: dpo_loss([0.0], [0.0], [0.0], [0.0], 0.0), lambda: dpo_loss([0.0], [0.0], [0.0], [0.0], -1.0),
                lambda: dpo_loss([0.0, 1.0], [0.0], [0.0], [0.0], 0.1), lambda: dpo_loss([], [], [], [], 0.1)):
        try:
            bad()
        except ValueError:
            continue
        raise AssertionError("a non-positive beta, arrays of different lengths, or no pairs should raise ValueError")
    print("exercise 3 looks right")


if __name__ == "__main__":
    _try("exercise 3", _check_dpo_loss)

## 8. Exercise 4: `dpo_coefficients()`

Training needs the loss's gradient. Because the loss depends on the policy only through the two
log-likelihoods `lp_w` and `lp_l` of each pair, the chain rule splits the gradient in two. With the
margin `m = beta * ((lp_w - ref_w) - (lp_l - ref_l))` and the loss `mean over pairs of -log sigmoid(m)`:

* the derivative of `-log sigmoid(m)` with respect to `m` is for you to work out (hint 1 says how);
* `m` depends on `lp_w` with slope `+beta` and on `lp_l` with slope `-beta`.

So, per pair, the loss's derivative with respect to the chosen answer's log-likelihood is a **negative
number** (raise it to lower the loss) and with respect to the rejected answer's a **positive** one, equal
in size, and that size is **large when the margin is wrong or small, close to zero once the pair is already
well separated**. Do not forget that the loss is a *mean* over the pairs.

`dpo_coefficients(margins, beta)` returns the two arrays `(c_w, c_l)`: the derivative of the mean loss
with respect to each pair's `lp_w` and `lp_l`. Compute `sigmoid(-m)` without overflow for a margin of
plus or minus a thousand: the safe form you used in the last exercise is the tool again.

<details><summary>💡 Hint 1 — what to think about</summary>

Sketch the chain rule for one pair: loss, then margin, then log-likelihood. Differentiate `-log sigmoid(m)`
with respect to `m` first (the derivative of a sigmoid is itself times one minus itself). Which signs does the
margin give to the two log-likelihoods? What does averaging over the batch do to each pair's share?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

One weight per pair, the sigmoid of the negated margin, times beta, divided by the number of pairs.
The chosen answer's coefficient is that weight with a minus sign, the rejected answer's without.
Check the margin of zero by hand: the weight there is one half.
</details>

In [ ]:
def dpo_coefficients(margins, beta):
    """Derivative of the MEAN DPO loss with respect to each pair's chosen and rejected log-likelihood.

    c_w_i = d loss / d lp_w_i and c_l_i = d loss / d lp_l_i, where n is the number of pairs and the loss is their mean.
    Nothing may overflow for margins of plus or minus a thousand. `margins` may be a list, a tuple or an array.

    Example (one pair with margin 0: the weight is sigmoid(0) = 1/2):
        >>> dpo_coefficients(np.array([0.0]), 0.5)
        (array([-0.25]), array([0.25]))

    Returns:
        (c_w, c_l): two float arrays with one entry per pair.
    """
    # YOUR CODE HERE
    raise NotImplementedError("implement dpo_coefficients")


def _check_dpo_coefficients() -> None:
    c_w, c_l = dpo_coefficients([0.0], 0.5)
    assert np.allclose(c_w, [-0.25], rtol=0, atol=1e-12) and np.allclose(c_l, [0.25], rtol=0, atol=1e-12), (
        f"at margin 0 the weight is beta * sigmoid(0) = 0.25 for one pair; got c_w {np.asarray(c_w).tolist()}, c_l {np.asarray(c_l).tolist()}. The chosen answer's coefficient is negative, the rejected one's positive")
    m = np.array([-2.0, 0.0, 3.0, 1.5])
    c_w, c_l = dpo_coefficients(m, 0.2)
    want = 0.2 / (1.0 + np.exp(m)) / 4
    assert np.allclose(c_w, -want, rtol=0, atol=1e-12) and np.allclose(c_l, want, rtol=0, atol=1e-12), (
        f"got c_w {np.round(c_w, 5).tolist()}, expected {np.round(-want, 5).tolist()}: the weight is proportional to beta and to 1 / (number of pairs), and it FALLS as the margin grows")
    with np.errstate(over="raise", invalid="raise", divide="raise"):
        c_w, c_l = dpo_coefficients(np.array([-1000.0, 1000.0]), 1.0)
    assert np.allclose(c_l, [0.5, 0.0], rtol=0, atol=1e-12), (
        f"margins of -1000 and +1000 should give weights 1/2 and 0 for two pairs; got {np.asarray(c_l).tolist()}. Compute sigmoid(-m) as exp(-logaddexp(0, m)) so exp() never overflows")
    # the coefficients really are the derivative of the loss
    rng = np.random.default_rng(1)
    lp_w, lp_l, ref_w, ref_l = rng.normal(size=(4, 5))
    loss0, margins = dpo_loss(lp_w, lp_l, ref_w, ref_l, 0.3)
    c_w, c_l = dpo_coefficients(margins, 0.3)
    step = 1e-6
    bumped = lp_w.copy()
    bumped[2] += step
    numeric = (dpo_loss(bumped, lp_l, ref_w, ref_l, 0.3)[0] - loss0) / step
    assert abs(numeric - c_w[2]) < 1e-5, f"c_w[2] = {c_w[2]:.6f} but a finite difference of dpo_loss gives {numeric:.6f}: check the 1/n and the sign"
    print("exercise 4 looks right")


if __name__ == "__main__":
    _try("exercise 4", _check_dpo_coefficients, needs=("exercise 3",))

## 9. Exercise 5: the backward pass of a log-likelihood

The coefficients say how the loss changes with each *sequence's* log-likelihood. The model's backward
pass (section 2) wants the gradient with respect to its *logits*. The missing link is the gradient of
one sequence's log-likelihood with respect to its logits, and you know it from T03-L03: at each counted
position it is **the one-hot of the target minus the softmax of the logits**. (Cross-entropy's gradient is the same
thing with the opposite sign.) DPO re-weights it:

`d loss / d logits[i, t, :]  =  coef[i] * mask[i, t] * ( one_hot(target[i, t]) - softmax(logits[i, t]) )`

Here `coef[i]` is the derivative of the loss with respect to sequence `i`'s log-likelihood, which is exactly
what `dpo_coefficients` returned (chosen sequences first, then rejected ones, in one batch). Positions
the mask excludes get a zero gradient. This single re-weighting is the whole difference between
supervised fine-tuning and DPO as far as the model's backward pass is concerned: SFT gives every
counted position the same sign, DPO gives each sequence its own signed weight.

`sequence_logprob_backward(logits, targets, mask, coef)` returns the array of shape `(B, T, V)`. Keep
the softmax stable (subtract the row maximum) and do not modify the inputs.

<details><summary>💡 Hint 1 — what to think about</summary>

Which arrays have shape (B,), (B, T) and (B, T, V)? How do you make a per-sequence number multiply
every position and every symbol of that sequence? What goes at positions with mask 0?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Build the softmax of the shifted logits, build an array of zeros of the same shape and write a 1 at each
target symbol (`np.put_along_axis`), subtract, then multiply by a weight made from coef and mask with the
right axes added.
</details>

In [ ]:
def sequence_logprob_backward(logits, targets, mask, coef):
    """Gradient of sum_i coef[i] * logprob_i with respect to the logits, where logprob_i is `sequence_logprob`'s value.

    Args:
        logits: (B, T, V); targets: (B, T) ints; mask: (B, T) 0/1; coef: (B,) the weight of each sequence.

    At a counted position (mask 1) the gradient row is coef[i] * (one_hot(target) - softmax(logits)); at an
    excluded position it is zero.

    Example (one counted position, two symbols, equal scores, target is symbol 0, weight 2):
        >>> sequence_logprob_backward(np.zeros((1, 1, 2)), np.array([[0]]), np.array([[1.0]]), np.array([2.0]))
        array([[[ 1., -1.]]])

    Returns:
        a float array shaped like `logits`; the inputs are not modified.
    """
    # YOUR CODE HERE
    raise NotImplementedError("implement sequence_logprob_backward")


def _check_sequence_logprob_backward() -> None:
    got = sequence_logprob_backward(np.zeros((1, 1, 2)), np.array([[0]]), np.array([[1.0]]), np.array([2.0]))
    assert got.shape == (1, 1, 2) and np.allclose(got, [[[1.0, -1.0]]], rtol=0, atol=1e-12), (
        f"one position, equal scores, target 0, weight 2 should give [1, -1]; got {got.tolist()}. It is coef * (one_hot - softmax), so the target's entry is positive for a positive coef")
    rng = np.random.default_rng(2)
    z = rng.normal(size=(3, 4, 5))
    t = rng.integers(0, 5, size=(3, 4))
    m = np.array([[1, 1, 0, 1], [0, 1, 1, 1], [1, 0, 0, 1]], dtype=float)
    coef = np.array([0.5, -1.5, 2.0])
    got = sequence_logprob_backward(z, t, m, coef)
    assert got.shape == z.shape, f"the result must have the shape of the logits {z.shape}; got {got.shape}"
    assert np.all(got[0, 2] == 0) and np.all(got[2, 1] == 0), "positions with mask 0 must get an exactly zero gradient"
    for q in ((0, 1, 3), (1, 0, 2), (2, 3, 0)):
        want = None
        b, i, v = q
        step = 1e-6

        def total(zz):
            return float((coef * sequence_logprob(zz, t, m)).sum())
        zp = z.copy()
        zp[b, i, v] += step
        zm = z.copy()
        zm[b, i, v] -= step
        want = (total(zp) - total(zm)) / (2 * step)
        assert abs(want - got[q]) < 1e-6, f"entry {q}: a finite difference of coef-weighted sequence_logprob gives {want:.6f}, yours is {got[q]:.6f}: check the sign, the mask and the per-sequence weight"
    big = np.array([[[10000.0, 0.0, 0.0]]])
    with np.errstate(over="raise", invalid="raise", divide="raise"):
        out = sequence_logprob_backward(big, np.array([[1]]), np.array([[1.0]]), np.array([1.0]))
    assert np.isfinite(out).all(), "logits of 10000 gave a non-finite gradient: subtract the row maximum before exponentiating"
    before = z.copy()
    sequence_logprob_backward(z, t, m, coef)
    assert np.array_equal(z, before), "your function changed the logits it was given; build a new array instead"
    print("exercise 5 looks right")


if __name__ == "__main__":
    _try("exercise 5", _check_sequence_logprob_backward, needs=("exercise 2",))

## 10. Putting it together, and proving it

`preference_loss_and_grads` below is **given**. It stacks the chosen and rejected answers into one
batch, runs the model forward, calls your `sequence_logprob`, your `dpo_loss` and `dpo_coefficients`,
turns the coefficients into logit gradients with your `sequence_logprob_backward`, and hands that to the
model's backward pass. Nothing else is DPO-specific. (A `kind="ipo"` option, used in exercise 8, swaps
the loss; ignore it for now.)

That is a long chain of hand-derived pieces, so the cell after it checks the **whole gradient against
finite differences**: perturb single parameters of a model that is not at the reference (otherwise every
margin is zero and the check is blind to the margin's dependence), and compare `(loss(+h) - loss(-h)) / 2h`
with the gradient the chain produces. If a piece is wrong, the sign pattern of the failures says which.

In [ ]:
def preference_loss_and_grads(params, chosen, rejected, ref_w, ref_l, beta, kind="dpo"):
    """One DPO (or IPO) step's loss and the gradient with respect to every parameter. Given; calls your exercises."""
    n = len(chosen)
    tokens = np.concatenate([chosen, rejected])
    logits, cache = forward(params, tokens[:, :-1])
    logits.setflags(write=False)
    targets, mask = tokens[:, 1:], response_mask(tokens)
    lp = sequence_logprob(logits, targets, mask)
    lp_w, lp_l = lp[:n], lp[n:]
    if kind == "dpo":
        loss, margins = dpo_loss(lp_w, lp_l, ref_w, ref_l, beta)
        c_w, c_l = dpo_coefficients(margins, beta)
    else:
        loss, c_w, c_l = ipo_loss(lp_w, lp_l, ref_w, ref_l, beta)
    d_logits = sequence_logprob_backward(logits, targets, mask, np.concatenate([c_w, c_l]))
    return loss, backward(params, cache, d_logits)


def _gradient_check() -> None:
    if not _ready("REF", "REF_W", "TRAIN_PAIRS"):
        return
    rng = np.random.default_rng([SEED, 9])
    params = {k: v + 0.05 * rng.normal(size=v.shape) for k, v in REF.items()}     # not at the reference: margins differ from zero
    pairs = {k: v[:6] for k, v in TRAIN_PAIRS.items() if k in ("chosen", "rejected")}
    ref_w, ref_l = REF_W[:6], REF_L[:6]
    _, grads = preference_loss_and_grads(params, pairs["chosen"], pairs["rejected"], ref_w, ref_l, BETA)
    worst, failed = 0.0, []
    for name, array in params.items():
        for q in rng.integers(0, array.size, size=5):
            old = array.flat[q]
            array.flat[q] = old + 1e-5
            up = preference_loss_and_grads(params, pairs["chosen"], pairs["rejected"], ref_w, ref_l, BETA)[0]
            array.flat[q] = old - 1e-5
            down = preference_loss_and_grads(params, pairs["chosen"], pairs["rejected"], ref_w, ref_l, BETA)[0]
            array.flat[q] = old
            numeric, analytic = (up - down) / 2e-5, grads[name].flat[q]
            slack = abs(numeric - analytic) / (1e-8 + 1e-4 * max(abs(numeric), abs(analytic)))
            worst = max(worst, slack)
            if slack > 1.0 and name not in failed:
                failed.append(name)
    print("the whole DPO gradient against central finite differences, 5 coordinates of each parameter array, batch of 6 pairs:")
    print("  every coordinate within the tolerance:", worst <= 1.0)
    assert worst <= 1.0, (f"the gradient disagrees with finite differences for {failed}. If only the embeddings and positions are wrong the backward "
                          "pass is fine and the logit gradient is not; if every array is wrong suspect the sign or the 1/n of exercise 4, or the mask of exercise 5")


if __name__ == "__main__":
    _try("gradient check", _gradient_check, needs=("exercise 2", "exercise 3", "exercise 4", "exercise 5"))

## 11. Training: DPO against SFT on the chosen answers

Two ways to use the same pairs, with the same starting model, learning rate, number of updates and batch
size:

* **SFT on the chosen answers**: ordinary maximum likelihood on the chosen response of every pair, ignoring
  the rejected one. The rejected answers are thrown away.
* **DPO**: the loss of section 6, which uses both answers and the reference model's likelihoods of them.

The training loop below is given. During DPO training it also reports, every so often, the loss on
the batch and, on **held-out pairs** (held-out prompts, never trained on), the share of pairs whose
implicit reward margin is positive (pair accuracy) and the mean margin. After training, each policy
writes answers for the held-out prompts and the truth scores them: true quality, length, filler
tokens, and how many are cut off at the cap. The numbers are means over several training seeds (the cell
prints how many; the seed changes the order of the batches, and the pairs are the same).

In [ ]:
def pair_report(params, pairs, ref_w, ref_l, beta):
    """Log-likelihoods of a pair set under `params`, and the DPO margins (uses your exercises 2 and 3)."""
    lp = []
    for key in ("chosen", "rejected"):
        tokens = pairs[key]
        logits, _ = forward(params, tokens[:, :-1])
        lp.append(sequence_logprob(logits, tokens[:, 1:], response_mask(tokens)))
    loss, margins = dpo_loss(lp[0], lp[1], ref_w, ref_l, beta)
    return {"lp_w": lp[0], "lp_l": lp[1], "margins": margins, "loss": loss}


def train_preference(ref_params, pairs, ref_w, ref_l, beta, kind="dpo", steps=POLICY_STEPS, lr=POLICY_LR, seed=0, watch=None, watch_every=50):
    params = {k: v.copy() for k, v in ref_params.items()}
    state, rng, n = adam_init(params), np.random.default_rng([SEED, 100 + seed]), len(pairs["chosen"])
    history = []
    for step in range(1, steps + 1):
        idx = rng.integers(0, n, size=PAIR_BATCH)
        loss, grads = preference_loss_and_grads(params, pairs["chosen"][idx], pairs["rejected"][idx], ref_w[idx], ref_l[idx], beta, kind)
        adam_step(params, grads, state, step, lr)
        if watch is not None and step % watch_every == 0:
            history.append((step, loss, watch(params)))
    return params, history


def train_sft_on_chosen(ref_params, pairs, steps=POLICY_STEPS, lr=POLICY_LR, seed=0):
    params = {k: v.copy() for k, v in ref_params.items()}
    n = len(pairs["chosen"])
    fit_nll(params, lambda rng: pairs["chosen"][rng.integers(0, n, size=PAIR_BATCH)], steps, lr, seed=100 + seed)
    return params


def _held_out_watch(beta):
    def watch(params):
        rep = pair_report(params, HELD_PAIRS, HELD_REF_W, HELD_REF_L, beta)
        return float((rep["margins"] > 0).mean()), float(rep["margins"].mean())
    return watch


def _train_dpo() -> None:
    global DPO_RUNS
    if not _ready("REF", "REF_W", "HELD_REF_W", "TRAIN_PAIRS", "HELD_PAIRS"):
        return
    print(f"DPO, beta = {BETA}, {POLICY_STEPS} updates of {PAIR_BATCH} pairs, {len(SEEDS)} training seeds; held-out pairs measured along the way for seed 0:")
    runs, rows = [], []
    for seed in SEEDS:
        params, history = train_preference(REF, TRAIN_PAIRS, REF_W, REF_L, BETA, seed=seed, watch=_held_out_watch(BETA) if seed == 0 else None)
        runs.append(params)
        if seed == 0:
            rows = [[step, f"{loss:.3f}", f"{100 * acc:.1f}%", f"{mean:.3f}"] for step, loss, (acc, mean) in history]
    _table(["update", "batch loss", "held-out pair accuracy", "held-out mean margin"], rows)
    DPO_RUNS = runs


def _train_sft() -> None:
    global SFT_RUNS
    if not _ready("REF", "TRAIN_PAIRS"):
        return
    SFT_RUNS = [train_sft_on_chosen(REF, TRAIN_PAIRS, seed=seed) for seed in SEEDS]
    print(f"SFT on the chosen answers: {len(SFT_RUNS)} training seeds, {POLICY_STEPS} updates of {PAIR_BATCH} chosen answers each, done.")


def _compare_policies() -> None:
    global POLICIES
    if not _ready("DPO_RUNS", "SFT_RUNS", "ref_eval"):
        return
    POLICIES = {"sft": SFT_RUNS, "dpo": DPO_RUNS}
    stats = {"reference": [ref_eval]}
    for name in ("sft", "dpo"):
        stats[name] = [evaluate_policy(p, seed=60 + i) for i, p in enumerate(POLICIES[name])]
    label = {"reference": "reference (SFT on noisy answers)", "sft": "SFT on the chosen answers", "dpo": f"DPO, beta = {BETA}"}
    print(f"answers to the {len(ref_eval['prompts'])} held-out prompts (mean over {len(SEEDS)} training seeds; the reference is one model):")
    table = []
    for name in ("reference", "sft", "dpo"):
        mean = lambda key: np.mean([s[key] for s in stats[name]])
        table.append([label[name], f"{mean('quality'):.2f}", f"{mean('length'):.2f}", f"{mean('fillers'):.2f}", f"{100 * mean('capped'):.1f}%"])
    _table(["policy", "true quality (of 4)", "length", "filler tokens", "cut off at the cap"], table)
    dq = np.mean([s["quality"] for s in stats["dpo"]]) - ref_eval["quality"]
    dl = np.mean([s["length"] for s in stats["dpo"]]) - ref_eval["length"]
    sl = np.mean([s["length"] for s in stats["sft"]]) - ref_eval["length"]
    print(f"\nrelative to the reference, DPO changed true quality by {dq:+.2f} letters and length by {dl:+.2f} tokens; SFT on the chosen answers changed length by {sl:+.2f} tokens.")
    longer = dl > 0.5
    better = dq > 0
    print("DPO's answers are", "longer" if longer else "not much longer",
          "and", "more correct" if better else "not more correct", "than the reference's.")
    if longer and better:
        extra = np.mean([s["fillers"] for s in stats["dpo"]]) - ref_eval["fillers"]
        print(f"Of the {dl:+.2f} extra tokens, {extra:+.2f} are filler tokens, which the truth scores at zero: the correctness gain is real and so is the length the labeller rewarded.")


if __name__ == "__main__":
    _try("train DPO", _train_dpo, needs=("exercise 2", "exercise 3", "exercise 4", "exercise 5"))
    _try("train SFT", _train_sft)
    _try("compare policies", _compare_policies, needs=("exercise 2", "exercise 3", "exercise 4", "exercise 5"))

## 12. Exercise 6: the chosen answer got less likely

DPO's loss only asks for a **larger margin** between the chosen and the rejected answer: the *gap* of their
log-ratios. Nothing in the loss asks the chosen answer's own likelihood to rise, and the gradient lowers
the rejected answer's likelihood at the same time as it raises the chosen's. Two papers (`claims.yaml`)
report this as a documented failure mode of DPO: **the preferred answer becomes less likely while the
margin goes up**. One of them points at datasets in which the two answers of a pair differ little, which
is where lowering the rejected answer can carry the chosen one with it.

`displacement_counts(lp_w, lp_l, ref_w, ref_l)` counts, over pairs, how often each thing happened. Return a
dict with exactly these integer entries, each a strict comparison (equality is not "down" and not "up"):

* `"pairs"`: the number of pairs;
* `"margin_up"`: pairs where `(lp_w - ref_w) - (lp_l - ref_l) > 0`;
* `"chosen_down"`: pairs where `lp_w < ref_w`;
* `"rejected_down"`: pairs where `lp_l < ref_l`;
* `"displaced"`: pairs where the margin went up **and** the chosen answer went down.

<details><summary>💡 Hint 1 — what to think about</summary>

Four boolean arrays over pairs, and the fifth count is the "and" of two of them. Does a pair whose chosen
answer is exactly as likely as before count as down? What type should each count have?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Compute the change of each answer's log-likelihood from the reference, the margin as the difference of those
changes, then each comparison against zero, then sum the booleans (and convert to int).
</details>

In [ ]:
def displacement_counts(lp_w, lp_l, ref_w, ref_l):
    """Count pairs by what happened to the margin and to each answer's likelihood.

    Returns a dict with the integer entries "pairs", "margin_up", "chosen_down", "rejected_down" and "displaced"
    as described above; every comparison is strict. The four inputs may be lists, tuples or arrays.

    Example (two pairs: the first raises the margin but lowers the chosen answer; the second does neither):
        >>> displacement_counts(np.array([-3.0, -1.0]), np.array([-9.0, -1.0]), np.array([-2.0, -1.0]), np.array([-4.0, -1.0]))
        {'pairs': 2, 'margin_up': 1, 'chosen_down': 1, 'rejected_down': 1, 'displaced': 1}
    """
    # YOUR CODE HERE
    raise NotImplementedError("implement displacement_counts")


def _check_displacement_counts() -> None:
    got = displacement_counts([-3.0, -1.0], [-9.0, -1.0], [-2.0, -1.0], [-4.0, -1.0])
    want = {"pairs": 2, "margin_up": 1, "chosen_down": 1, "rejected_down": 1, "displaced": 1}
    assert got == want, f"the worked example (given as plain lists) should give {want}; got {got}. Convert the inputs with np.asarray first"
    got = displacement_counts(np.array([-1.0, -2.0, -3.0, 0.5]), np.array([-5.0, -2.0, -9.0, 1.5]), np.array([-2.0, -2.0, -2.0, 0.0]), np.array([-3.0, -2.0, -2.0, 1.0]))
    want = {"pairs": 4, "margin_up": 2, "chosen_down": 1, "rejected_down": 2, "displaced": 1}
    assert got == want, (f"four pairs (a pair whose answers did not change counts as neither up nor down, and a margin of exactly zero is not up) should give {want}; got {got}")
    assert all(type(v) is int for v in got.values()), f"every count must be a plain Python int, got {[type(v).__name__ for v in got.values()]}"
    assert displacement_counts(np.array([]), np.array([]), np.array([]), np.array([])) == {"pairs": 0, "margin_up": 0, "chosen_down": 0, "rejected_down": 0, "displaced": 0}, "no pairs should give all zeros"
    print("exercise 6 looks right")


if __name__ == "__main__":
    _try("exercise 6", _check_displacement_counts)

In [ ]:
def _show_displacement() -> None:
    if not _ready("POLICIES", "REF_W", "HELD_REF_W"):
        return
    print("what happened to the likelihoods, per pair, on the pairs each policy was trained on and on held-out pairs (training seed 0):")
    rows, chosen_change = [], []
    for name, params in (("SFT on chosen", POLICIES["sft"][0]), ("DPO", POLICIES["dpo"][0])):
        for split, pairs, rw, rl in (("train", TRAIN_PAIRS, REF_W, REF_L), ("held-out", HELD_PAIRS, HELD_REF_W, HELD_REF_L)):
            rep = pair_report(params, pairs, rw, rl, BETA)
            c = displacement_counts(rep["lp_w"], rep["lp_l"], rw, rl)
            chosen_change.append(float((rep["lp_w"] - rw).mean()))
            rows.append([name, split, c["pairs"], c["margin_up"], c["chosen_down"], c["rejected_down"], c["displaced"],
                         f"{(rep['lp_w'] - rw).mean():+.2f}", f"{(rep['lp_l'] - rl).mean():+.2f}"])
    _table(["policy", "split", "pairs", "margin up", "chosen down", "rejected down", "displaced", "mean change chosen", "mean change rejected"], rows)
    held = rows[3]                                   # DPO on held-out pairs
    print(f"\nafter DPO, on held-out pairs the margin rose for {held[3]} of {held[2]} pairs, and for {held[6]} of them the chosen answer"
          f" became LESS likely than under the reference ({held[4]} pairs had a less likely chosen answer in all).")
    print("on held-out pairs the chosen answers' mean log-likelihood changed by", f"{float(held[7]):+.2f}", "nats under DPO and by",
          f"{float(rows[1][7]):+.2f}", "nats under SFT on the chosen answers, which never looks at the rejected ones.")
    print("DPO lowered the chosen answers' likelihood on average:", chosen_change[3] < 0)
    print("This table cannot say how much of any such fall is the toy and how much is the loss. Two things can lower a chosen answer's likelihood: the stored"
          " answers are samples of the NOISY reference, so a policy that has moved towards correct, longer answers gives those exact strings less probability;"
          " and the loss never asks for the chosen answer's own likelihood to be kept, only for the gap to widen. The SFT row is a partial control, not a clean one:"
          " its loss does ask for the chosen answers' likelihood to rise, and its chosen-down count shows how often the first reason alone is enough.")


if __name__ == "__main__":
    _try("displacement", _show_displacement, needs=("exercise 2", "exercise 3", "exercise 4", "exercise 5", "exercise 6"))

## 13. Exercise 7: how far did the policy travel?

The `beta` in the objective prices distance from the reference. A way to measure the distance a trained
policy actually travelled is the **KL divergence from the policy to the reference on the policy's own
answers**: sample answers from the policy, and average `log pi(y|x) - log pi_ref(y|x)` over them. Each
term is one number per sampled answer, and their mean estimates `KL(pi || pi_ref)`.

An estimate from a finite sample comes with an error bar. `kl_estimate(lp_pol, lp_ref)` returns
`(mean, standard_error)` of the per-answer log-ratios, with the standard error of the mean using the
sample standard deviation with **n - 1** in the denominator (`ddof=1`): `std / sqrt(n)`. Raise
`ValueError` for fewer than two answers or arrays of different lengths. The order of the subtraction matters:
policy minus reference, on answers drawn from the policy. (One caveat: the sampler of section 3 never draws the
separator or padding symbol, so the answers are not exactly samples from the policy; the probability mass printed
there is the size of that effect.)

<details><summary>💡 Hint 1 — what to think about</summary>

What is the sample you average over, and which of the two likelihoods comes first in the subtraction?
What does a standard error divide by, and why does n - 1 appear in the sample standard deviation?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

One array of log-ratios, its mean, its standard deviation with one degree of freedom removed, divided by
the square root of the number of answers. Validate the lengths and the count first.
</details>

In [ ]:
def kl_estimate(lp_pol, lp_ref):
    """Monte-Carlo estimate of KL(policy || reference) from answers SAMPLED FROM THE POLICY, with its standard error.

    `lp_pol[i]` and `lp_ref[i]` are the policy's and the reference's log-likelihood of the same sampled answer i.
    The estimate is the mean of lp_pol - lp_ref; the standard error is the sample standard deviation (ddof=1)
    divided by sqrt(n). Raise ValueError for fewer than two answers or arrays of different lengths.
    The inputs may be lists, tuples or arrays.

    Example (log-ratios 1, 0, 2, 0: mean 0.75):
        >>> mean, se = kl_estimate(np.array([-1.0, -2.0, -3.0, -4.0]), np.array([-2.0, -2.0, -5.0, -4.0]))
        >>> mean, round(se, 4)
        (0.75, 0.4787)

    Returns:
        (mean, standard_error) as two Python floats.
    """
    # YOUR CODE HERE
    raise NotImplementedError("implement kl_estimate")


def _check_kl_estimate() -> None:
    mean, se = kl_estimate([-1.0, -2.0, -3.0, -4.0], [-2.0, -2.0, -5.0, -4.0])
    assert abs(mean - 0.75) < 1e-12, f"the mean of the log-ratios 1, 0, 2, 0 is 0.75; got {mean}. Policy minus reference, not the other way round"
    assert abs(se - np.sqrt(2.75 / 3 / 4)) < 1e-12, (
        f"the standard error should be std(ddof=1)/sqrt(n) = {np.sqrt(2.75 / 3 / 4):.6f}; got {se}: divide the sum of squares by n - 1 for the standard deviation, then the standard deviation by sqrt(n)")
    assert type(mean) is float and type(se) is float, f"return two Python floats, not {type(mean).__name__} and {type(se).__name__}"
    mean, se = kl_estimate(np.array([0.5, 0.5, 0.5]), np.array([0.0, 0.0, 0.0]))
    assert mean == 0.5 and se == 0.0, f"identical log-ratios have no spread: expected (0.5, 0.0), got {(mean, se)}"
    for bad in (lambda: kl_estimate([1.0], [1.0]), lambda: kl_estimate([1.0, 2.0], [1.0]), lambda: kl_estimate([], [])):
        try:
            bad()
        except ValueError:
            continue
        raise AssertionError("fewer than two answers, or arrays of different lengths, should raise ValueError")
    print("exercise 7 looks right")


if __name__ == "__main__":
    _try("exercise 7", _check_kl_estimate)

## 14. The effect of beta

Small `beta` prices distance cheaply, so the policy travels; large `beta` prices it dearly. The cell
trains DPO at several values of `beta` from the same start and the same pairs, samples answers from each
trained policy (a fresh draw for every one, not the held-out prompts' pairs), and measures with your
`kl_estimate` how far each has travelled, next to what it did to quality and length. Each row is one
training run with seed 0 and its own draw of answers, so the row for the beta of section 11 will not
equal that section's mean over three seeds.

In [ ]:
BETAS = (0.02, 0.1, 0.5)


def _kl_of(params: dict):
    ev = evaluate_policy(params, seed=70)
    lp = []
    for model in (params, REF):
        logits, _ = forward(model, ev["tokens"][:, :-1])
        lp.append(sequence_logprob(logits, ev["tokens"][:, 1:], response_mask(ev["tokens"])))
    return ev, kl_estimate(lp[0], lp[1])


def _beta_sweep() -> None:
    global SWEEP
    if not _ready("POLICIES", "REF_W", "HELD_REF_W"):
        return
    SWEEP = {}
    rows = []
    for beta in BETAS:
        params, _ = train_preference(REF, TRAIN_PAIRS, REF_W, REF_L, beta, seed=0)
        ev, (kl, se) = _kl_of(params)
        rep = pair_report(params, HELD_PAIRS, HELD_REF_W, HELD_REF_L, beta)
        SWEEP[beta] = {"kl": kl, "length": ev["length"], "quality": ev["quality"]}
        rows.append([f"{beta:g}", f"{kl:.2f} ± {se:.2f}", f"{100 * (rep['margins'] > 0).mean():.1f}%", f"{ev['quality']:.2f}", f"{ev['length']:.2f}"])
    params = POLICIES["sft"][0]
    ev, (kl, se) = _kl_of(params)
    rows.append(["SFT on chosen", f"{kl:.2f} ± {se:.2f}", "n/a", f"{ev['quality']:.2f}", f"{ev['length']:.2f}"])
    _table(["beta", "KL to reference (nats)", "held-out pair accuracy", "true quality (of 4)", "length"], rows)
    kls = [SWEEP[b]["kl"] for b in BETAS]
    print("\nthe KL estimate", "falls" if all(a > b for a, b in zip(kls, kls[1:])) else "does not fall steadily",
          "as beta grows through", ", ".join(f"{b:g}" for b in BETAS) + ".")
    farthest = max(BETAS, key=lambda b: SWEEP[b]['kl'])
    spread = max(SWEEP[b]['length'] for b in BETAS) - min(SWEEP[b]['length'] for b in BETAS)
    print(f"the policy that travelled farthest has beta = {farthest:g}. Answer lengths in tokens:",
          ", ".join(f"beta {b:g}: {SWEEP[b]['length']:.2f}" for b in BETAS) + f"; they differ by at most {spread:.2f} tokens. Each is one training run and one draw of answers, so"
          " compare lengths across betas by the large differences only: a difference of a fraction of a token is inside what the choice of seed and of draw can move.")


if __name__ == "__main__":
    _try("beta sweep", _beta_sweep, needs=("exercise 2", "exercise 3", "exercise 4", "exercise 5", "exercise 7"))

## 15. Exercise 8: IPO, a loss with a target

DPO's loss has no finish line: for a pair that is already separated, the gradient shrinks but never
reaches zero, and the margin can keep growing. The IPO paper (`claims.yaml`) replaces the logistic loss
with a **squared error towards a fixed target** for the gap between the policy's and the reference's
log-ratios:

`h = (lp_w - ref_w) - (lp_l - ref_l)`,   `loss = mean over pairs of ( h - 1 / (2 * beta) ) ^ 2`

Here `beta` plays the role of the paper's regularisation strength, so the target gap is `1 / (2 * beta)` nats
and the pull towards it switches off when it is reached. Whether a training run gets there is a separate
question, which section 16 measures. (The paper's own symbol is tau; this lesson
reuses the name `beta` so you can compare the two losses at the same value.)

`ipo_loss(lp_w, lp_l, ref_w, ref_l, beta)` returns `(loss, c_w, c_l)`: the loss, and the derivative of the
**mean** loss with respect to each pair's `lp_w` and `lp_l`, in the same convention as exercise 4, so the
given training loop can use it in place of DPO's. Derive the derivative yourself: differentiate the
square, then apply the chain rule through `h`, which depends on `lp_w` with slope `+1` and on `lp_l` with
slope `-1`. Raise `ValueError` as `dpo_loss` does.

<details><summary>💡 Hint 1 — what to think about</summary>

The derivative of `(h - target)^2` with respect to `h` is a multiple of `(h - target)`. What are the slopes
of `h` in `lp_w` and in `lp_l`? Where does the "mean over pairs" enter?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Compute `h` per pair, subtract the target, square and average for the loss. For the coefficients, two times
the error over the number of pairs for the chosen answer, and its negative for the rejected one.
</details>

In [ ]:
def ipo_loss(lp_w, lp_l, ref_w, ref_l, beta):
    """The IPO loss (squared error of the log-ratio gap towards 1 / (2 beta)) and its coefficients.

    h_i = (lp_w_i - ref_w_i) - (lp_l_i - ref_l_i);   loss = mean_i of (h_i - 1/(2 beta))^2.
    c_w_i = d loss / d lp_w_i and c_l_i = d loss / d lp_l_i, for the MEAN loss.
    Raise ValueError if beta <= 0, if any of the four arrays differs in length from another, or if there are no pairs.
    The four inputs may be lists, tuples or arrays; convert them with `np.asarray` first.

    Example (h = 2 and a target of 1 at beta = 0.5: error 1):
        >>> loss, c_w, c_l = ipo_loss(np.array([-1.0]), np.array([-3.0]), np.array([-2.0]), np.array([-2.0]), 0.5)
        >>> loss, c_w, c_l
        (1.0, array([2.]), array([-2.]))

    Returns:
        (loss, c_w, c_l): a Python float and two float arrays with one entry per pair.
    """
    # YOUR CODE HERE
    raise NotImplementedError("implement ipo_loss")


def _check_ipo_loss() -> None:
    loss, c_w, c_l = ipo_loss([-1.0], [-3.0], [-2.0], [-2.0], 0.5)
    assert abs(loss - 1.0) < 1e-12 and np.allclose(c_w, [2.0], rtol=0, atol=1e-12) and np.allclose(c_l, [-2.0], rtol=0, atol=1e-12), (
        f"h = 2, target 1/(2*0.5) = 1, so the loss is 1 and the coefficients are +2 and -2; got {loss}, {np.asarray(c_w).tolist()}, {np.asarray(c_l).tolist()}")
    loss, c_w, c_l = ipo_loss(np.array([1.0, 0.0]), np.array([0.0, 0.0]), np.array([0.0, 0.0]), np.array([0.0, 0.0]), 1.0)
    assert abs(loss - 0.25) < 1e-12 and np.allclose(c_w, [0.5, -0.5], rtol=0, atol=1e-12), (
        f"two pairs with h = 1 and 0 at beta = 1 (target 0.5): errors 0.5 and -0.5, loss 0.25, c_w [0.5, -0.5]; got {loss}, {np.asarray(c_w).tolist()}. The loss is a MEAN over pairs, so each coefficient carries a 1/n")
    loss, c_w, c_l = ipo_loss(np.array([0.5]), np.array([0.0]), np.array([0.0]), np.array([0.0]), 1.0)
    assert loss == 0.0 and c_w[0] == 0.0 and c_l[0] == 0.0, "a pair exactly at the target gap has no loss and no gradient"
    rng = np.random.default_rng(3)
    lp_w, lp_l, ref_w, ref_l = rng.normal(size=(4, 5))
    loss0, c_w, c_l = ipo_loss(lp_w, lp_l, ref_w, ref_l, 0.7)
    bumped = lp_l.copy()
    bumped[3] += 1e-6
    numeric = (ipo_loss(lp_w, bumped, ref_w, ref_l, 0.7)[0] - loss0) / 1e-6
    assert abs(numeric - c_l[3]) < 1e-4, f"c_l[3] = {c_l[3]:.6f} but a finite difference of ipo_loss gives {numeric:.6f}: the rejected answer's coefficient has the opposite sign of the chosen one's"
    for bad in (lambda: ipo_loss([0.0], [0.0], [0.0], [0.0], 0.0), lambda: ipo_loss([0.0, 1.0], [0.0], [0.0], [0.0], 0.1), lambda: ipo_loss([], [], [], [], 0.1)):
        try:
            bad()
        except ValueError:
            continue
        raise AssertionError("a non-positive beta, arrays of different lengths, or no pairs should raise ValueError")
    print("exercise 8 looks right")


if __name__ == "__main__":
    _try("exercise 8", _check_ipo_loss)

## 16. IPO against DPO at the same beta

The same pairs, the same start, the same number of updates, `beta = 0.1`. IPO's target gap is
`1 / (2 * beta)` nats. The first cell reports, on the training pairs, the mean log-ratio gap `h` each
policy ended at after the usual number of updates, next to what each did to the chosen answers' likelihood,
the distance travelled, and the answers' quality and length (training seed 0 and a fresh draw of answers,
so the DPO row will not equal the three-seed means of section 11). The second cell trains both losses for
twice as long and prints the gap along the way, to see whether either one stops. A caution before you
read it: a pair with a *wrong* label cannot be fitted by a model that has to be right about other pairs, so
IPO's mean gap need not reach its target; the cell prints where it ends up rather than assuming.

In [ ]:
def _ipo_vs_dpo() -> None:
    if not _ready("POLICIES", "REF_W", "HELD_REF_W"):
        return
    rows, raw, target = [], [], 1.0 / (2.0 * BETA)
    for name, kind in (("DPO", "dpo"), ("IPO", "ipo")):
        params = POLICIES["dpo"][0] if kind == "dpo" else train_preference(REF, TRAIN_PAIRS, REF_W, REF_L, BETA, kind="ipo", seed=0)[0]
        rep = pair_report(params, TRAIN_PAIRS, REF_W, REF_L, BETA)
        h = (rep["lp_w"] - REF_W) - (rep["lp_l"] - REF_L)
        ev, (kl, se) = _kl_of(params)
        rows.append([name, f"{h.mean():.2f}", f"{(rep['lp_w'] - REF_W).mean():+.2f}", f"{kl:.2f} ± {se:.2f}", f"{ev['quality']:.2f}", f"{ev['length']:.2f}"])
        raw.append({"gap": float(h.mean()), "kl": kl, "quality": ev["quality"], "length": ev["length"]})
    _table(["loss", "mean gap h (nats)", "mean change of chosen", "KL to reference", "true quality (of 4)", "length"], rows)
    print(f"\nIPO's target gap at beta = {BETA} is {target:.1f} nats.")
    print("DPO's mean gap is", "larger" if raw[0]["gap"] > raw[1]["gap"] else "not larger", "than IPO's, and its KL to the reference is",
          "larger." if raw[0]["kl"] > raw[1]["kl"] else "not larger.")
    print("IPO's answers are", "shorter" if raw[1]["length"] < raw[0]["length"] else "not shorter", "than DPO's and",
          "less" if raw[1]["quality"] < raw[0]["quality"] else "not less", "correct "
          f"({raw[1]['length'] - raw[0]['length']:+.2f} tokens and {raw[1]['quality'] - raw[0]['quality']:+.2f} letters from single runs: read the direction, not the size).")


def _gap_watch(params: dict):
    rep = pair_report(params, TRAIN_PAIRS, REF_W, REF_L, BETA)
    return float(((rep["lp_w"] - REF_W) - (rep["lp_l"] - REF_L)).mean())


def _ipo_longer() -> None:
    if not _ready("REF_W", "TRAIN_PAIRS"):
        return
    steps, every = 2 * POLICY_STEPS, POLICY_STEPS // 2
    target = 1.0 / (2.0 * BETA)
    print(f"the same two losses trained for {steps} updates (twice as long), mean gap h on the training pairs along the way (target for IPO: {target:.1f} nats):")
    gaps = {}
    for name, kind in (("DPO", "dpo"), ("IPO", "ipo")):
        _, history = train_preference(REF, TRAIN_PAIRS, REF_W, REF_L, BETA, kind=kind, seed=0, steps=steps, watch=_gap_watch, watch_every=every)
        gaps[name] = [value for _, _, value in history]
    marks = list(range(every, steps + 1, every))
    _table(["loss"] + [f"after {m} updates" for m in marks], [[name] + [f"{v:.2f}" for v in gaps[name]] for name in gaps])
    last = {name: gaps[name][-1] - gaps[name][-2] for name in gaps}
    print(f"\nover the last {every} updates DPO's mean gap changed by {last['DPO']:+.2f} nats and IPO's by {last['IPO']:+.2f}.")
    below = gaps["IPO"][-1] < 0.9 * target
    flat = abs(last["IPO"]) < 0.5 * abs(last["DPO"])
    print("IPO's gap changed", "less than half as much as" if flat else "about as much as, or more than,", "DPO's over those updates, and it ended",
          "well below" if below else "near or above", f"its target of {target:.1f} nats.")
    if below and flat:
        print("So the target is not what holds it there: pairs labelled the wrong way round (this world has them) and the model's limits both pull the gap down,"
              " and this table cannot say how much each does.")
    elif below:
        print("So it is still on its way: the run is too short to show where it stops.")
    else:
        print("So in this run it stopped at its target.")

if __name__ == "__main__":
    _try("IPO against DPO", _ipo_vs_dpo, needs=("exercise 2", "exercise 3", "exercise 4", "exercise 5", "exercise 7", "exercise 8"))
    _try("IPO over longer training", _ipo_longer, needs=("exercise 2", "exercise 3", "exercise 4", "exercise 5", "exercise 8"))

## 17. What to take away, and what the toy hides

Everything above is computed by the cells you ran; this one only reads those numbers back. Where a
sentence depends on how a number came out, the cell chooses the wording from the number.

In [ ]:
def _bottom_line() -> None:
    if not _ready("POLICIES", "HELD_REF_W", "ref_eval"):
        return
    dpo = [evaluate_policy(p, seed=60 + i) for i, p in enumerate(POLICIES["dpo"])]
    sft = [evaluate_policy(p, seed=60 + i) for i, p in enumerate(POLICIES["sft"])]
    ql = np.mean([s["quality"] for s in dpo]) - ref_eval["quality"]
    ll = np.mean([s["length"] for s in dpo]) - ref_eval["length"]
    lq = np.mean([s["quality"] for s in sft]) - ref_eval["quality"]
    ls = np.mean([s["length"] for s in sft]) - ref_eval["length"]
    print(f"DPO moved true quality by {ql:+.2f} letters and length by {ll:+.2f} tokens against the reference; "
          f"SFT on the chosen answers moved them by {lq:+.2f} and {ls:+.2f}.")
    rep = pair_report(POLICIES["dpo"][0], HELD_PAIRS, HELD_REF_W, HELD_REF_L, BETA)
    c = displacement_counts(rep["lp_w"], rep["lp_l"], HELD_REF_W, HELD_REF_L)
    print(f"On held-out pairs the margin rose for {c['margin_up']} of {c['pairs']}, and the chosen answer became less likely for {c['chosen_down']} of them.")
    gained_len = ll > 0.5
    gained_q = ql > 0
    if gained_len and gained_q:
        print("So this run mixes a real gain with a spurious one: the model got more correct and longer at once, and the pairs cannot say which of the"
              " two the preference was about. Only an audit like that of section 4, which the truth and a length-blind control make possible here, separates them.")
    elif gained_len:
        print("So this run shows the spurious gain without the real one: longer answers, no more correct.")
    else:
        print("So in this run the length bias did not carry over into longer answers; the audit of section 4 still shows it in the labels.")
    print("\nThe toy hides three things, which you should keep in mind when you read a real DPO result:")
    print("  * the truth is known here, so quality and length can be separated; with real human or model labels they cannot be, without the audit;")
    print("  * the model has a few thousand parameters and ten symbols, so 'likelihood displacement' has little room to move probability onto"
          " answers of a different meaning; with a large model it can;")
    print("  * the pairs were sampled from the reference and used once, offline; methods that sample again from the current policy"
          " (online DPO) see the model's own drift, which this lesson cannot.")


if __name__ == "__main__":
    _try("what to take away", _bottom_line, needs=("exercise 2", "exercise 3", "exercise 4", "exercise 5", "exercise 6"))

## 18. Common mistakes

* **The sign of the chosen answer's coefficient.** A positive coefficient on the chosen answer minimises the
  wrong thing: the loss falls when the chosen answer gets *less* likely. The finite-difference check of
  section 10 catches it; reading the margin curve (it falls instead of rising) would too.
* **Averaging instead of summing the token log-probabilities.** The ratio of whole-sequence probabilities is what the
  derivation uses. A per-token mean changes what is optimised and silently removes the length effect you came to measure.
* **Forgetting the mask.** Prompt tokens and padding then enter the log-likelihood. The reference and the policy
  share the prompt, so most of the error cancels in a margin, which is why this bug hides.
* **Computing the reference log-likelihoods with the policy as it trains.** The reference is frozen: its numbers are
  computed once, before training, from the model you started with.
* **Reading pair accuracy as quality.** The labeller's preference includes the planted length bonus; a high
  accuracy on its labels says the model learned the labeller, not the task.

## 19. Self-check

1. The policy equals the reference at the first update. Which statement is true of the first DPO step?
   - (a) Every margin is positive, because in every pair the chosen answer is the better one of the two
   - (b) The loss is smallest at this moment, because nothing has moved yet and the margins have nothing to correct
   - (c) Every margin is zero, so all pairs weigh the same and chosen answers go up, rejected ones down
   - (d) The gradient is zero, because the policy and the reference agree on the likelihood of every answer

2. After DPO training, the margin on held-out pairs is positive for most pairs, and yet the average
   log-likelihood of the chosen answers is lower than the reference's. What is the best reading?
   - (a) A bug: a loss that raises the margin cannot lower the likelihood of the chosen answers in any run
   - (b) Expected: the loss constrains the gap between two log-ratios, not the chosen answer's own level
   - (c) The learning rate was too small for the chosen answers' likelihood to move in the right direction
   - (d) The reference model was not frozen, so its own likelihoods drifted along with the policy's

3. You halve `beta` and train for the same number of updates with Adam. Which change would you expect, other things equal?
   - (a) The policy stays closer to the reference, because a smaller beta means a stronger constraint on distance
   - (b) Nothing changes, because Adam normalises the size of the gradient and so cancels the scale of beta
   - (c) The margin saturates earlier, so the policy moves less before the loss stops asking for more
   - (d) The policy can travel farther, because distance is priced more cheaply

4. A preference set is audited on the pairs whose two answers differ in length. In them the chosen answer is the
   longer one far more than half of the time, while a control labeller that ignores length, run on the very same
   answers, picks the longer one only about half of the time. What does that most plausibly mean?
   - (a) Length is moving the labels, apart from quality
   - (b) The audit code must contain a bug somewhere in its counting
   - (c) The labeller answers at random whenever the lengths differ
   - (d) Longer answers are simply the better ones in every dataset

<details><summary>Answers</summary>

Answers come with this lesson's worked solution when you enrol on Synapsa.
</details>

In [ ]:
_MARKS = {"passed": "✅", "failed": "❌", "not started": "⏳"}


def _progress_board() -> None:
    """One line per exercise, from the latest run of its check, then the tally."""
    width = max(len(", ".join(funcs)) for funcs in _EXERCISES.values())
    print("progress board")
    for label, funcs in _EXERCISES.items():
        state = _STATUS.get(label, "not started")
        print(f"  {_MARKS[state]} {label:<12} {', '.join(funcs):<{width}}  {state}")
    done = sum(_STATUS.get(label) == "passed" for label in _EXERCISES)
    print(f"\n{done} of {len(_EXERCISES)} exercises complete")
    failing = [label for label in _EXERCISES if _STATUS.get(label) == "failed"]
    if failing:
        print("failing right now: " + ", ".join(failing) + ". Each one printed what went "
              "wrong in its own cell above, and every exercise has hints you can open.")
    elif done < len(_EXERCISES):
        print("work top to bottom: every exercise has hints you can open above its code.")
    broken = [label for label, state in _STATUS.items() if label not in _EXERCISES and state == "failed"]
    if broken:
        print("a demo that runs your code failed: " + ", ".join(broken) + ". Its own cell above says why; "
              "a check that passes can still miss a case a demo reaches.")

## What you built, and where it goes next

You followed the derivation of DPO from the KL-regularised reward objective, turned it into four small pieces
(a stable sequence log-likelihood, a loss with implicit-reward margins, its gradient coefficients and the
backward pass that re-weights per-sequence gradients), proved the chain against finite differences, and then
measured what the method does: the margin it raises, the chosen answers it can make less likely, the length it
can reward, and the distance it travels at each beta. IPO showed that a different loss changes the finish
line, not the machinery.

Left out on purpose, with sources in `claims.yaml` (as of 19 October 2026): SimPO (the average log-likelihood as the reward, no
reference model), KTO (learning from a binary good-or-bad signal instead of pairs), ORPO (an odds-ratio
penalty added to supervised fine-tuning, no reference model), and online DPO (new answers sampled
from the current policy). Each changes one of the choices you made here, and you now have the harness to
measure what the change buys.

In [ ]:
# Your progress board. Every check is re-run here, quietly, against your code as it stands
# now: each one already printed its feedback in its own cell above, so the board is
# current even if you edited an exercise and did not re-run its check.
if __name__ == "__main__":
    with contextlib.redirect_stdout(io.StringIO()):
        for _name, _check, _needs in (
                ("exercise 1", _check_win_counts, ()),
                ("exercise 2", _check_sequence_logprob, ()),
                ("exercise 3", _check_dpo_loss, ()),
                ("exercise 4", _check_dpo_coefficients, ("exercise 3",)),
                ("exercise 5", _check_sequence_logprob_backward, ("exercise 2",)),
                ("exercise 6", _check_displacement_counts, ()),
                ("exercise 7", _check_kl_estimate, ()),
                ("exercise 8", _check_ipo_loss, ())):
            _try(_name, _check, needs=_needs)
    _progress_board()
    # A stub you have not reached yet is not a failure. A check that ran and came back wrong
    # is: in a script or under CI it ends this run non-zero, rather than letting a green exit
    # code paper over it. Inside a notebook kernel the board above has already said so, in a
    # line rather than a traceback at the foot of the page.
    if _FAILED_CHECKS and "ipykernel" not in sys.modules:
        raise SystemExit("checks failed: " + ", ".join(dict.fromkeys(_FAILED_CHECKS)))

<!-- COMMONS NOTICE v1 · generated by tools/notebooks.py · do not edit by hand -->
---
**Synapsa Commons** · © 2026 RealAI · licensed under [CC BY-NC-SA
4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)

**You may** use this lesson to learn and to teach, and copy, fork, share and adapt it.

**You must** credit "Synapsa Commons by RealAI" with a link to
https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials,
say what you changed, and share anything you adapt under this same licence.

**You may not** use it, or anything adapted from it, in a way primarily intended for
commercial advantage or payment: for example selling it, charging for a course, bootcamp or
training built on it, or packaging it into a paid product or service. For a commercial
licence, contact [RealAI](https://www.realai.eu/contact).

Third-party material in this lesson keeps its own licence, named in `assets/SOURCE.md` or
`claims.yaml`. The Synapsa name and logo belong to RealAI and are not licensed. This summary
is not the licence: the [legal
code](https://creativecommons.org/licenses/by-nc-sa/4.0/legalcode) governs.